In [1]:
import random
import math
from datetime import datetime, timedelta, date
from faker import Faker

import mysql.connector
fake = Faker("pt_BR")
random.seed(42)  # garante que os dados são reproduzíveis

In [2]:
# configurações banco de dados

DB_CONFIG = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "root", #apenas removi a senha aqui para vc nao ver
    "database": "apex_telemetria",
    "charset": "utf8mb4"
}

VOLUME = "medio" #"pequeno" ; "medio" ; "grande"

VOLUMES = {
    "pequeno": {"temporadas": 1, "eventos_por_temp": 5,  "voltas_corrida": 40, "telemetria": False},
    "medio":   {"temporadas": 2, "eventos_por_temp": 10, "voltas_corrida": 60, "telemetria": True},
    "grande":  {"temporadas": 3, "eventos_por_temp": 20, "voltas_corrida": 71, "telemetria": True},
}

CFG = VOLUMES[VOLUME]

In [3]:
def connect():
    return mysql.connector.connect(**DB_CONFIG)
 
def new_cur(conn):
    """Cursor com buffered=True evita 'Unread result found'."""
    return conn.cursor(buffered=True)
 
def rnd(a, b, dec=2):
    return round(random.uniform(a, b), dec)
 
def rnd_int(a, b):
    return random.randint(a, b)

In [4]:
#dados de referencia
PAISES = [
    ("BRA","Brasil","America do Sul"),   ("GBR","Reino Unido","Europa"),
    ("ITA","Italia","Europa"),           ("AUT","Austria","Europa"),
    ("DEU","Alemanha","Europa"),         ("NLD","Paises Baixos","Europa"),
    ("MCO","Monaco","Europa"),           ("ESP","Espanha","Europa"),
    ("FRA","Franca","Europa"),           ("USA","Estados Unidos","America do Norte"),
    ("AUS","Australia","Oceania"),       ("JPN","Japao","Asia"),
    ("MEX","Mexico","America do Norte"), ("SGP","Singapura","Asia"),
    ("ARE","Emirados Arabes","Asia"),    ("BEL","Belgica","Europa"),
    ("HUN","Hungria","Europa"),          ("BAH","Bahrein","Asia"),
    ("CAN","Canada","America do Norte"), ("CHN","China","Asia"),
    ("FIN","Finlandia","Europa"),        ("DNK","Dinamarca","Europa"),
    ("NZL","Nova Zelandia","Oceania"),
]
 
EQUIPES_DATA = [
    ("Mercedes-AMG PETRONAS",       "Mercedes",    "GBR", "Toto Wolff",       2010),
    ("Scuderia Ferrari",            "Ferrari",     "ITA", "Fred Vasseur",     1929),
    ("Oracle Red Bull Racing",      "Red Bull",    "AUT", "Christian Horner", 2005),
    ("McLaren F1 Team",             "McLaren",     "GBR", "Andrea Stella",    1963),
    ("Aston Martin Aramco F1 Team", "Aston Martin","GBR", "Mike Krack",       2021),
    ("BWT Alpine F1 Team",          "Alpine",      "FRA", "Oliver Oakes",     1977),
    ("Williams Racing",             "Williams",    "GBR", "James Vowles",     1977),
    ("MoneyGram Haas F1 Team",      "Haas",        "USA", "Ayao Komatsu",     2016),
    ("Visa Cash App RB F1 Team",    "Racing Bulls","ITA", "Laurent Mekies",   2006),
    ("Stake F1 Team Kick Sauber",   "Sauber",      "DEU", "Mattia Binotto",   1993),
]
 
PILOTOS_DATA = [
    # (codigo, nome, sobrenome, iso_pais, nascimento, numero, equipe_idx)
    ("HAM","Lewis",    "Hamilton",  "GBR","1985-01-07", 44, 1),
    ("RUS","George",   "Russell",   "GBR","1998-02-15", 63, 0),
    ("VER","Max",      "Verstappen","NLD","1997-09-30",  1, 2),
    ("NOR","Lando",    "Norris",    "GBR","1999-11-13",  4, 3),
    ("LEC","Charles",  "Leclerc",   "MCO","1997-10-16", 16, 1),
    ("PIA","Oscar",    "Piastri",   "AUS","2001-04-06", 81, 3),
    ("ALO","Fernando", "Alonso",    "ESP","1981-07-29", 14, 4),
    ("STR","Lance",    "Stroll",    "CAN","1998-10-29", 18, 4),
    ("GAS","Pierre",   "Gasly",     "FRA","1996-02-07", 10, 5),
    ("OCO","Esteban",  "Ocon",      "FRA","1996-09-17", 31, 5),
    ("ALB","Alexander","Albon",     "GBR","1996-03-23", 23, 6),
    ("SAR","Logan",    "Sargeant",  "USA","2000-12-31",  2, 6),
    ("MAG","Kevin",    "Magnussen", "DNK","1992-10-05", 20, 7),
    ("HUL","Nico",     "Hulkenberg","DEU","1987-08-19", 27, 7),
    ("TSU","Yuki",     "Tsunoda",   "JPN","2000-05-11", 22, 8),
    ("LAW","Liam",     "Lawson",    "NZL","2002-02-11", 30, 8),
    ("BOT","Valtteri", "Bottas",    "FIN","1989-08-28", 77, 9),
    ("ZHO","Guanyu",   "Zhou",      "CHN","1999-05-30", 24, 9),
    ("ANT","Andrea",   "Antonelli", "ITA","2006-08-25", 12, 0),
    ("BEA","Oliver",   "Bearman",   "GBR","2005-05-08", 87, 1),
]
 
CIRCUITOS_DATA = [
    # nome, apelido, iso, cidade, comp_m, curvas, sentido, alt, lat, lon
    ("Autodromo Jose Carlos Pace",    "Interlagos","BRA","Sao Paulo",       4309,15,"anti-horario", 785,-23.7036,-46.6975),
    ("Circuit de Monaco",             "Monaco",    "MCO","Monte Carlo",     3337,19,"horario",        7, 43.7347,  7.4205),
    ("Silverstone Circuit",           "Silverstone","GBR","Silverstone",    5891,18,"horario",      153, 52.0786, -1.0169),
    ("Autodromo Nazionale Monza",     "Monza",     "ITA","Monza",           5793,11,"horario",      162, 45.6156,  9.2811),
    ("Circuit de Barcelona-Catalunya","Barcelona", "ESP","Barcelona",       4657,16,"horario",      115, 41.5700,  2.2611),
    ("Hungaroring",                   "Budapest",  "HUN","Budapest",        4381,14,"horario",      264, 47.5789, 19.2486),
    ("Circuit Gilles Villeneuve",     "Montreal",  "CAN","Montreal",        4361,14,"anti-horario",  13, 45.5000,-73.5226),
    ("Suzuka Circuit",                "Suzuka",    "JPN","Suzuka",          5807,18,"misto",          40, 34.8431,136.5411),
    ("Marina Bay Street Circuit",     "Singapura", "SGP","Singapura",       4940,23,"horario",        15,  1.2914,103.8638),
    ("Yas Marina Circuit",            "Abu Dhabi", "ARE","Abu Dhabi",       5281,16,"anti-horario",    3, 24.4672, 54.6031),
    ("Spa-Francorchamps",             "Spa",       "BEL","Spa",             7004,20,"horario",       401, 50.4372,  5.9715),
    ("Bahrain International Circuit", "Bahrein",   "BAH","Sakhir",          5412,15,"horario",         7, 26.0325, 50.5106),
    ("Shanghai International Circuit","Xangai",    "CHN","Xangai",          5451,16,"horario",         5, 31.3389,121.2198),
    ("Circuit of The Americas",       "Austin",    "USA","Austin",          5513,20,"anti-horario",  161, 30.1328,-97.6411),
    ("Autodromo Hermanos Rodriguez",  "Mexico",    "MEX","Cidade do Mexico",4304,17,"horario",      2240, 19.4042,-99.0907),
]
 
TIPOS_SENSOR_DATA = [
    ("RPM",          "Rotacao Motor",             "motor",       "rpm",    0,  20000, 200),
    ("TEMP_MOT",     "Temperatura Motor",         "motor",       "C",      0,    200,  10),
    ("PRESS_OLEO",   "Pressao do Oleo",           "motor",       "bar",    0,     15,  50),
    ("CONSUMO",      "Consumo Combustivel",       "motor",       "kg/lap", 0,      5,  10),
    ("COMBUSTIVEL",  "Nivel Combustivel",         "motor",       "kg",     0,    110,   5),
    ("VELOC",        "Velocidade Linear",         "aerodinamica","km/h",   0,    400,  50),
    ("ACCEL_LONG",   "Aceleracao Longitudinal",   "suspensao",   "g",     -8,      8, 200),
    ("ACCEL_LAT",    "Aceleracao Lateral",        "suspensao",   "g",     -7,      7, 200),
    ("TEMP_PNEU_DD", "Temp Pneu Diant Dir",       "pneu",        "C",      0,    150,  10),
    ("TEMP_PNEU_DE", "Temp Pneu Diant Esq",       "pneu",        "C",      0,    150,  10),
    ("TEMP_PNEU_TD", "Temp Pneu Tras Dir",        "pneu",        "C",      0,    150,  10),
    ("TEMP_PNEU_TE", "Temp Pneu Tras Esq",        "pneu",        "C",      0,    150,  10),
    ("PRESS_PNEU_DD","Pressao Pneu Diant Dir",    "pneu",        "psi",   15,     35,  10),
    ("PRESS_PNEU_DE","Pressao Pneu Diant Esq",    "pneu",        "psi",   15,     35,  10),
    ("PRESS_PNEU_TD","Pressao Pneu Tras Dir",     "pneu",        "psi",   15,     35,  10),
    ("PRESS_PNEU_TE","Pressao Pneu Tras Esq",     "pneu",        "psi",   15,     35,  10),
    ("TEMP_FREIO_DD","Temp Freio Diant Dir",      "freio",       "C",      0,   1200,  20),
    ("TEMP_FREIO_DE","Temp Freio Diant Esq",      "freio",       "C",      0,   1200,  20),
    ("TEMP_FREIO_TD","Temp Freio Tras Dir",       "freio",       "C",      0,   1200,  20),
    ("TEMP_FREIO_TE","Temp Freio Tras Esq",       "freio",       "C",      0,   1200,  20),
    ("PRESS_FREIO",  "Pressao Linha Freio",       "freio",       "bar",    0,    200, 100),
    ("VOLT_ERS",     "Tensao Bateria ERS",        "eletrico",    "V",      0,   1000,  10),
    ("CARGA_ERS",    "Carga Bateria ERS",         "eletrico",    "pct",    0,    100,  10),
    ("ACELERADOR",   "Posicao Acelerador",        "motor",       "pct",    0,    100, 100),
    ("FREIO_PEDAL",  "Posicao Freio",             "freio",       "pct",    0,    100, 100),
    ("MARCHA",       "Marcha Engatada",           "transmissao", "n",     -1,      8,  50),
    ("DRS",          "Status DRS",               "aerodinamica","bool",   0,      1,  10),
    ("GPS_LAT",      "Latitude GPS",             "gps",         "graus", -90,    90,  50),
    ("GPS_LON",      "Longitude GPS",            "gps",         "graus",-180,   180,  50),
    ("FREQ_CARD",    "Frequencia Cardiaca",       "piloto",      "bpm",   40,    220,   1),
    ("TEMP_COCKPIT", "Temperatura Cockpit",       "piloto",      "C",      0,     60,   5),
]
 
PONTOS_F1   = [25,18,15,12,10,8,6,4,2,1]
COMPOSTOS_S = ["slick_soft","slick_medium","slick_hard"]
 
TEMPO_BASE = {
    "Interlagos":73000,"Monaco":72000,"Silverstone":85000,"Monza":80000,
    "Barcelona":75000,"Budapest":76000,"Montreal":73000,"Suzuka":89000,
    "Singapura":95000,"Abu Dhabi":84000,"Spa":103000,"Bahrein":90000,
    "Xangai":93000,"Austin":94000,"Mexico":78000,
}
 
HABILIDADE = {
    "VER":0,"NOR":200,"HAM":100,"LEC":150,"PIA":300,"RUS":350,
    "ALO":400,"BEA":450,"GAS":500,"ALB":600,"STR":700,"HUL":650,
    "TSU":750,"LAW":800,"BOT":900,"MAG":950,"ZHO":1000,"OCO":850,
    "ANT":1100,"SAR":1200,
}
 
ENGENHEIROS = [
    "Peter Bonnington","Riccardo Adami","Gianpiero Lambiase","Will Joseph",
    "Tom Stallard","Chris Cronin","Gaetan Jego","Francesco Nenci",
    "Andrew Shovlin","Xevi Pujolar","Alan Permane","Bradley Joyce",
]
 
MECANICOS = [
    "Joao Silva","Carlos Mendes","Hans Muller","Luigi Ferrari",
    "James Cooper","Pierre Dubois","Mario Rossi","Ahmed Al-Rashid",
    "Pedro Costa","Ralf Schmidt","Marco Bianchi","Steve Thompson",
]

In [5]:
# funcoes de geração de dados
def gerar_tempo_volta(base_ms, composto, idade_pneu, combustivel_kg, habilidade=0, sc=False):
    t = base_ms + habilidade
    t += {"slick_soft":-400,"slick_medium":0,"slick_hard":300,
          "intermediario":3000,"chuva":7000}.get(composto, 0)
    t += {"slick_soft":35,"slick_medium":18,"slick_hard":8,
          "intermediario":25,"chuva":40}.get(composto, 15) * idade_pneu
    t += combustivel_kg * 30
    t += rnd_int(-200, 200)
    if sc:
        t += rnd_int(15000, 25000)
    return max(int(t), base_ms - 2000)
 
 
def estrategia_pit(total_voltas, condicao):
    meio = total_voltas // 2
    if condicao in ("molhado","muito_molhado"):
        return random.choice([
            [(12,"chuva","intermediario"),(36,"intermediario","slick_medium")],
            [(18,"chuva","intermediario")],
        ])
    return random.choice([
        [(meio + rnd_int(-3,3), "slick_medium","slick_hard")],
        [(meio - 10,            "slick_soft",  "slick_hard")],
        [(total_voltas//3,      "slick_soft",  "slick_medium"),
         (total_voltas*2//3,    "slick_medium","slick_hard")],
    ])


In [6]:
# modulos de inserção de dados
def step_referencias(c):
    print("  -> paises, categorias, tipos_sensor, componentes, canais_telemetria")
 
    c.executemany(
        "INSERT IGNORE INTO paises (codigo_iso,nome,continente) VALUES (%s,%s,%s)",
        PAISES)
 
    c.execute(
        "INSERT IGNORE INTO categorias (nome,descricao) VALUES (%s,%s)",
        ("Formula 1","Campeonato Mundial FIA"))
 
    for row in TIPOS_SENSOR_DATA:
        c.execute(
            "INSERT IGNORE INTO tipos_sensor "
            "(codigo,nome,categoria,unidade,valor_min,valor_max,frequencia_hz) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s)", row)
 
    comps = [
        ("Motor de Combustao Interna","motor",         4000, None),
        ("Caixa de Cambio",           "caixa_cambio",  None,  6.0),
        ("MGU-H",                     "motor_energia",  None, None),
        ("MGU-K",                     "motor_energia",  None, None),
        ("Bateria ERS",               "baterias",       None, None),
        ("Turbocompressor",           "turbo",          None, None),
        ("Unidade Eletronica ECU",    "unidade_eletrica",None,None),
    ]
    c.executemany(
        "INSERT IGNORE INTO componentes "
        "(nome,tipo,vida_util_km,vida_util_horas) VALUES (%s,%s,%s,%s)", comps)
 
    # canais_telemetria
    c.execute("SELECT id_tipo_sensor, codigo, unidade FROM tipos_sensor")
    tipos = c.fetchall()
    grupos = {
        "RPM":"Motor","TEMP_MOT":"Motor","PRESS_OLEO":"Motor",
        "CONSUMO":"Motor","COMBUSTIVEL":"Motor","ACELERADOR":"Motor",
        "MARCHA":"Transmissao","VELOC":"Aerodinamica","DRS":"Aerodinamica",
        "ACCEL_LONG":"Dinamica","ACCEL_LAT":"Dinamica",
        "TEMP_PNEU_DD":"Pneus","TEMP_PNEU_DE":"Pneus",
        "TEMP_PNEU_TD":"Pneus","TEMP_PNEU_TE":"Pneus",
        "PRESS_PNEU_DD":"Pneus","PRESS_PNEU_DE":"Pneus",
        "PRESS_PNEU_TD":"Pneus","PRESS_PNEU_TE":"Pneus",
        "TEMP_FREIO_DD":"Freios","TEMP_FREIO_DE":"Freios",
        "TEMP_FREIO_TD":"Freios","TEMP_FREIO_TE":"Freios",
        "PRESS_FREIO":"Freios","FREIO_PEDAL":"Freios",
        "VOLT_ERS":"ERS","CARGA_ERS":"ERS",
        "GPS_LAT":"GPS","GPS_LON":"GPS",
        "FREQ_CARD":"Piloto","TEMP_COCKPIT":"Piloto",
    }
    for id_tipo, codigo, unidade in tipos:
        grupo  = grupos.get(codigo, "Geral")
        formula = f"AVG({codigo})" if ("TEMP" in codigo or "PRESS" in codigo) else f"MAX({codigo})"
        c.execute(
            "INSERT IGNORE INTO canais_telemetria "
            "(nome,grupo,id_tipo_sensor,formula_calculo,unidade,descricao) "
            "VALUES (%s,%s,%s,%s,%s,%s)",
            (f"Canal_{codigo}", grupo, id_tipo, formula, unidade,
             f"Canal de analise para o sensor {codigo}"))
 
 
# =============================================================================
#  STEP 2 - FABRICANTES, MOTORES, EQUIPES, PILOTOS
# =============================================================================
def step_equipes_pilotos(c):
    print("  -> fabricantes, motores, equipes, pilotos")
 
    fabs = [
        ("Mercedes-AMG HPP",        "GBR"),
        ("Scuderia Ferrari SpA",     "ITA"),
        ("Red Bull Powertrains",     "AUT"),
        ("McLaren Racing",           "GBR"),
        ("Renault Sport",            "FRA"),
        ("Honda Racing Corp",        "JPN"),
        ("Aston Martin Perf Tech",   "GBR"),
    ]
    fab_ids = {}
    for nome, iso in fabs:
        c.execute("SELECT id_pais FROM paises WHERE codigo_iso=%s", (iso,))
        row = c.fetchone()
        id_pais = row[0] if row else 1
        c.execute(
            "INSERT IGNORE INTO fabricantes (nome,id_pais) VALUES (%s,%s)",
            (nome, id_pais))
        c.execute(
            "SELECT id_fabricante FROM fabricantes WHERE nome=%s", (nome,))
        fab_ids[nome] = c.fetchone()[0]
 
    id_merc = fab_ids["Mercedes-AMG HPP"]
    id_ferr = fab_ids["Scuderia Ferrari SpA"]
    id_rb   = fab_ids["Red Bull Powertrains"]
    id_mcl  = fab_ids["McLaren Racing"]
 
    motores = [
        (id_merc,"M17-EQ",   "Mercedes M17 E Performance","hibrido",6,1600,1060,None),
        (id_ferr,"F067-26",  "Ferrari 067/26",             "hibrido",6,1600,1040,None),
        (id_rb,  "RBPTH002", "Honda RBPTH002",             "hibrido",6,1600,1020,None),
        (id_mcl, "M17-MCL",  "Mercedes M17 cliente",       "hibrido",6,1600,1050,None),
    ]
    for m in motores:
        c.execute(
            "INSERT IGNORE INTO motores "
            "(id_fabricante,codigo,nome,tipo,cilindros,cilindrada_cc,potencia_cv,torque_nm) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s,%s)", m)
 
    eq_ids = []
    for nome,apelido,iso,diretor,fund in EQUIPES_DATA:
        c.execute("SELECT id_pais FROM paises WHERE codigo_iso=%s", (iso,))
        row = c.fetchone(); id_pais = row[0] if row else 1
        c.execute(
            "INSERT IGNORE INTO equipes "
            "(nome,apelido,id_pais,diretor,ano_fundacao) VALUES (%s,%s,%s,%s,%s)",
            (nome,apelido,id_pais,diretor,fund))
        c.execute("SELECT id_equipe FROM equipes WHERE nome=%s", (nome,))
        eq_ids.append(c.fetchone()[0])
 
    pilot_ids = {}
    for cod,nome,sob,iso,nasc,num,eq_idx in PILOTOS_DATA:
        c.execute("SELECT id_pais FROM paises WHERE codigo_iso=%s", (iso,))
        row = c.fetchone(); id_pais = row[0] if row else 1
        c.execute(
            "INSERT IGNORE INTO pilotos "
            "(codigo_piloto,nome,sobrenome,id_pais,data_nascimento,numero_carro,id_equipe) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s)",
            (cod,nome,sob,id_pais,nasc,num,eq_ids[eq_idx]))
        c.execute(
            "SELECT id_piloto FROM pilotos WHERE codigo_piloto=%s", (cod,))
        pilot_ids[cod] = c.fetchone()[0]
 
    return eq_ids, pilot_ids, fab_ids
 
 
# =============================================================================
#  STEP 3 - CIRCUITOS
# =============================================================================
def step_circuitos(c):
    print("  -> circuitos")
    circ_ids = {}
    for nome,apelido,iso,cidade,comp,curv,sent,alt,lat,lon in CIRCUITOS_DATA:
        c.execute("SELECT id_pais FROM paises WHERE codigo_iso=%s", (iso,))
        row = c.fetchone(); id_pais = row[0] if row else 1
        c.execute(
            "INSERT IGNORE INTO circuitos "
            "(nome,apelido,id_pais,cidade,comprimento_m,numero_curvas,"
            "sentido,altitude_m,latitude,longitude) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
            (nome,apelido,id_pais,cidade,comp,curv,sent,alt,lat,lon))
        c.execute(
            "SELECT id_circuito FROM circuitos WHERE apelido=%s", (apelido,))
        circ_ids[apelido] = c.fetchone()[0]
    return circ_ids
 
 
# =============================================================================
#  STEP 4 - VEICULOS
# =============================================================================
def step_veiculos(c, eq_ids, fab_ids):
    print("  -> modelos_veiculo, veiculos")
 
    id_merc = fab_ids["Mercedes-AMG HPP"]
    id_ferr = fab_ids["Scuderia Ferrari SpA"]
    id_rb   = fab_ids["Red Bull Powertrains"]
    id_mcl  = fab_ids["McLaren Racing"]
    id_ren  = fab_ids["Renault Sport"]
    id_am   = fab_ids["Aston Martin Perf Tech"]
 
    modelos_raw = [
        (id_merc,"W17",2026),(id_ferr,"SF-26",2026),(id_rb,"RB22",2026),
        (id_mcl,"MCL40",2026),(id_ren,"A526",2026),(id_am,"AMR26",2026),
        (id_merc,"FW47",2026),(id_am,"VF-26",2026),(id_rb,"VCARB3",2026),
        (id_ferr,"C46",2026),
    ]
    mod_ids = []
    for id_fab,nome,ano in modelos_raw:
        c.execute(
            "INSERT IGNORE INTO modelos_veiculo "
            "(id_fabricante,nome,ano_modelo,categoria) VALUES (%s,%s,%s,'Formula 1')",
            (id_fab,nome,ano))
        c.execute(
            "SELECT id_modelo FROM modelos_veiculo WHERE nome=%s AND ano_modelo=%s",
            (nome,ano))
        mod_ids.append(c.fetchone()[0])
 
    siglas  = ["W17","SF26","RB22","MCL","ALP","AMR","FW","VF","VCB","C46"]
    numeros = [(63,44),(16,87),(1,30),(4,81),(10,31),(14,18),(23,2),(27,20),(22,30),(77,24)]
 
    veic_por_eq = {}
    for i,(eq_id,(n1,n2)) in enumerate(zip(eq_ids, numeros)):
        mid = mod_ids[i]
        veic_por_eq[eq_id] = []
        for j,num in enumerate([n1,n2]):
            chassi = f"{siglas[i]}-2026-{num:03d}-{rnd_int(1000,9999)}"
            try:
                c.execute(
                    "INSERT INTO veiculos "
                    "(id_modelo,id_equipe,numero_chassi,numero_carro,"
                    "data_fabricacao,km_total,status) "
                    "VALUES (%s,%s,%s,%s,'2026-01-15',0,'ativo')",
                    (mid,eq_id,chassi,num))
                veic_por_eq[eq_id].append(c.lastrowid)
            except Exception:
                c.execute(
                    "SELECT id_veiculo FROM veiculos WHERE numero_chassi=%s",
                    (chassi,))
                row = c.fetchone()
                if row:
                    veic_por_eq[eq_id].append(row[0])
    return veic_por_eq
 
 
# =============================================================================
#  STEP 5 - SENSORES
# =============================================================================
def step_sensores(c, veic_por_eq):
    print("  -> sensores")
    c.execute("SELECT id_tipo_sensor, codigo FROM tipos_sensor")
    tipos = c.fetchall()
 
    pos_map = {
        "RPM":"bloco motor","TEMP_MOT":"cabeca cilindro","PRESS_OLEO":"circuito oleo",
        "VELOC":"chassi central","ACCEL_LONG":"chassi central","ACCEL_LAT":"chassi central",
        "TEMP_PNEU_DD":"roda DD","TEMP_PNEU_DE":"roda DE",
        "TEMP_PNEU_TD":"roda TD","TEMP_PNEU_TE":"roda TE",
        "PRESS_PNEU_DD":"roda DD","PRESS_PNEU_DE":"roda DE",
        "PRESS_PNEU_TD":"roda TD","PRESS_PNEU_TE":"roda TE",
        "TEMP_FREIO_DD":"pastilha DD","TEMP_FREIO_DE":"pastilha DE",
        "TEMP_FREIO_TD":"pastilha TD","TEMP_FREIO_TE":"pastilha TE",
        "PRESS_FREIO":"cilindro mestre","VOLT_ERS":"bateria ERS","CARGA_ERS":"bateria ERS",
        "ACELERADOR":"pedal acelerador","FREIO_PEDAL":"pedal freio",
        "MARCHA":"caixa cambio","DRS":"asa traseira",
        "GPS_LAT":"antena GPS","GPS_LON":"antena GPS",
        "FREQ_CARD":"assento piloto","TEMP_COCKPIT":"painel cockpit",
        "CONSUMO":"sistema combustivel","COMBUSTIVEL":"tanque",
    }
 
    sensor_ids_veiculo = {}
    todos = [v for vs in veic_por_eq.values() for v in vs]
 
    for id_v in todos:
        sensor_ids_veiculo[id_v] = []
        for id_t, cod in tipos:
            ns  = f"SN-{id_v:03d}-{id_t:03d}-{rnd_int(100,9999)}"
            pos = pos_map.get(cod, "chassi")
            c.execute(
                "INSERT IGNORE INTO sensores "
                "(id_veiculo,id_tipo_sensor,numero_serie,posicao,data_instalacao,ativo) "
                "VALUES (%s,%s,%s,%s,'2026-01-10',1)",
                (id_v,id_t,ns,pos))
            if c.lastrowid:
                sensor_ids_veiculo[id_v].append(c.lastrowid)
 
        if not sensor_ids_veiculo[id_v]:
            c.execute(
                "SELECT id_sensor FROM sensores WHERE id_veiculo=%s LIMIT 10",
                (id_v,))
            sensor_ids_veiculo[id_v] = [r[0] for r in c.fetchall()]
 
    return sensor_ids_veiculo
 
 
# =============================================================================
#  STEP 6 - TEMPORADAS, EVENTOS, SESSOES, METEOROLOGIA
# =============================================================================
def step_temporadas_eventos(c, circ_ids, n_temp, n_eventos):
    print("  -> temporadas, eventos, sessoes, meteorologia")
    c.execute(
        "SELECT id_categoria FROM categorias WHERE nome='Formula 1'")
    id_cat = c.fetchone()[0]
 
    temp_ids = []
    for off in range(n_temp):
        ano = 2024 + off
        c.execute(
            "INSERT IGNORE INTO temporadas (id_categoria,ano,descricao) VALUES (%s,%s,%s)",
            (id_cat, ano, f"Campeonato Mundial F1 {ano}"))
        c.execute(
            "SELECT id_temporada FROM temporadas WHERE id_categoria=%s AND ano=%s",
            (id_cat,ano))
        temp_ids.append(c.fetchone()[0])
 
    circs = list(CIRCUITOS_DATA)
    random.shuffle(circs)
    circs = circs[:n_eventos]
 
    eventos_info = []  # (id_evento, id_sessao_corrida, apelido, condicao, lat, lon)
 
    for t_idx, id_temp in enumerate(temp_ids):
        data_ref = date(2024 + t_idx, 3, 1)
 
        for rodada,(nome_c,apelido,_,_,_,_,_,_,lat,lon) in enumerate(circs, 1):
            id_circ = circ_ids.get(apelido)
            if not id_circ:
                continue
 
            roll     = random.random()
            condicao = "seco" if roll<0.70 else ("umido" if roll<0.85 else "molhado")
            temp_ar  = rnd(18,35)
            temp_pi  = rnd(25,55)
            umid     = rnd(40,90)
            pres     = rnd(1005,1020)
            vento    = rnd(5,35)
 
            d_ini = data_ref
            d_fim = data_ref + timedelta(days=2)
 
            c.execute(
                "INSERT INTO eventos "
                "(id_temporada,id_circuito,nome,rodada,data_inicio,data_fim,status) "
                "VALUES (%s,%s,%s,%s,%s,%s,'concluido')",
                (id_temp, id_circ,
                 f"Grande Premio de {apelido} {2024+t_idx}",
                 rodada,
                 d_ini.strftime("%Y-%m-%d"), d_fim.strftime("%Y-%m-%d")))
            id_ev = c.lastrowid
 
            sessoes_tipos = [
                ("treino_livre_1", d_ini,              "14:00:00","16:00:00"),
                ("treino_livre_2", d_ini,              "18:00:00","19:00:00"),
                ("treino_livre_3", d_ini+timedelta(1), "13:00:00","14:00:00"),
                ("classificacao",  d_ini+timedelta(1), "16:00:00","17:00:00"),
                ("corrida",        d_fim,              "15:00:00","17:30:00"),
            ]
            id_corrida = None
            for tipo,dia,h0,h1 in sessoes_tipos:
                dt0 = datetime.strptime(f"{dia} {h0}", "%Y-%m-%d %H:%M:%S")
                dt1 = datetime.strptime(f"{dia} {h1}", "%Y-%m-%d %H:%M:%S")
                c.execute(
                    "INSERT INTO sessoes "
                    "(id_evento,tipo,numero,data_hora_inicio,data_hora_fim,"
                    "condicao_pista,temperatura_ar,temperatura_pista,"
                    "umidade_pct,pressao_hpa,velocidade_vento_kmh) "
                    "VALUES (%s,%s,1,%s,%s,%s,%s,%s,%s,%s,%s)",
                    (id_ev, tipo,
                     dt0.strftime("%Y-%m-%d %H:%M:%S"),
                     dt1.strftime("%Y-%m-%d %H:%M:%S"),
                     condicao,temp_ar,temp_pi,umid,pres,vento))
                if tipo == "corrida":
                    id_corrida = c.lastrowid
 
            # Meteorologia horaria no dia da corrida
            cond_str = ("sol" if condicao=="seco"
                        else ("chuva_leve" if condicao=="umido" else "chuva_forte"))
            for h in range(11, 18):
                ts = datetime(d_fim.year, d_fim.month, d_fim.day, h, 0)
                precip = rnd(0,5) if condicao != "seco" else 0.0
                c.execute(
                    "INSERT INTO meteorologia "
                    "(id_evento,timestamp,temperatura_ar,temperatura_pista,"
                    "umidade_pct,pressao_hpa,velocidade_vento_kmh,"
                    "direcao_vento_graus,precipitacao_mm,condicao) "
                    "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                    (id_ev, ts.strftime("%Y-%m-%d %H:%M:%S"),
                     temp_ar+rnd(-2,2), temp_pi+rnd(-3,3),
                     umid+rnd(-5,5), pres+rnd(-1,1),
                     vento+rnd(-5,5), rnd_int(0,359), precip, cond_str))
 
            eventos_info.append((id_ev, id_corrida, apelido, condicao, lat, lon))
            data_ref += timedelta(days=14)
 
    return temp_ids, eventos_info
 
 
# =============================================================================
#  STEP 7 - CORRIDAS COMPLETAS
#  (setups, voltas, setores, pit_stops, resultados, gps_posicoes,
#   telemetria, telemetria_volta_resumo, alertas_pista,
#   incidentes, falhas, relatorios_engenharia)
# =============================================================================
def step_corridas(c, eventos_info, pilot_ids, eq_ids, veic_por_eq,
                  sensor_ids_veiculo, total_voltas, fazer_telemetria):
 
    print(f"  -> corridas: {len(eventos_info)} eventos x {total_voltas} voltas x {len(pilot_ids)} pilotos")
 
    c.execute("SELECT id_piloto,id_equipe FROM pilotos")
    piloto_eq = {r[0]:r[1] for r in c.fetchall()}
 
    pilotos_lista = list(pilot_ids.items())  # [(cod, id), ...]
 
    pontos_acc = {pid:0 for pid in pilot_ids.values()}
    stats      = {pid:{"vitorias":0,"poles":0,"podios":0,"vr":0}
                  for pid in pilot_ids.values()}
 
    piloto_list_ids = list(pilot_ids.values())  # para calcular idx
 
    def get_veic(pid):
        eq = piloto_eq.get(pid)
        vs = veic_por_eq.get(eq, [])
        if not vs:
            return None
        idx = piloto_list_ids.index(pid) % len(vs) if pid in piloto_list_ids else 0
        return vs[idx]
 
    for ev_idx,(id_ev,id_sessao,apelido,condicao,circ_lat,circ_lon) in enumerate(eventos_info):
        print(f"     [{ev_idx+1:02d}/{len(eventos_info)}] {apelido} ({condicao})")
 
        base_ms = TEMPO_BASE.get(apelido, 85000)
 
        grid = list(pilotos_lista)
        random.shuffle(grid)
        pole_id = grid[0][1]
        stats[pole_id]["poles"] += 1
 
        sc_volta = rnd_int(10, total_voltas-15) if random.random()<0.4 else None
        sc_dur   = rnd_int(3,6) if sc_volta else 0
        abandonam = {pid for _,pid in pilotos_lista if random.random()<0.10}
 
        # ── SETUPS ──────────────────────────────────────────────────────────
        for cod,pid in pilotos_lista:
            id_v = get_veic(pid)
            if not id_v or not id_sessao:
                continue
            comp_d = random.choice(COMPOSTOS_S)
            comp_t = random.choice(COMPOSTOS_S)
            c.execute(
                "INSERT INTO setups_veiculo "
                "(id_veiculo,id_sessao,id_piloto,"
                "asa_dianteira_graus,asa_traseira_graus,"
                "altura_dianteira_mm,altura_traseira_mm,"
                "mola_dianteira_nm,mola_traseira_nm,"
                "composto_dianteiro,composto_traseiro,"
                "pressao_pneu_dd_psi,pressao_pneu_de_psi,"
                "pressao_pneu_td_psi,pressao_pneu_te_psi,"
                "balanco_freio_pct,carga_combustivel_kg,observacoes) "
                "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                (id_v,id_sessao,pid,
                 rnd(8,15),rnd(12,22),rnd(18,26),rnd(55,68),
                 rnd(80000,130000),rnd(90000,140000),
                 comp_d,comp_t,
                 rnd(21,25),rnd(21,25),rnd(19,23),rnd(19,23),
                 rnd(54,62),rnd(100,110),
                 fake.sentence(nb_words=6)))
 
        # ── VOLTAS ──────────────────────────────────────────────────────────
        resultados_temp = []
        id_voltas_piloto = {}
 
        for pos_larg,(cod,pid) in enumerate(grid, 1):
            id_v = get_veic(pid)
            if not id_v:
                continue
 
            hab      = HABILIDADE.get(cod, 800)
            pits     = estrategia_pit(total_voltas, condicao)
            pits_map = {volta:(ci,co) for volta,ci,co in pits}
 
            if condicao == "seco":
                comp = random.choice(["slick_soft","slick_medium"])
            elif condicao == "umido":
                comp = "intermediario"
            else:
                comp = "chuva"
 
            combustivel = rnd(105,110)
            idade_pneu  = 0
            pos_atual   = pos_larg
            voltas_info = []  # (id_volta_db, t_ms, pit_in, comp_in, comp_out)
            abnd_volta  = rnd_int(total_voltas//4, total_voltas-5) if pid in abandonam else None
 
            for vn in range(1, total_voltas+1):
                if abnd_volta and vn > abnd_volta:
                    break
 
                sc_ativo = bool(sc_volta and sc_volta <= vn <= sc_volta + sc_dur)
                pit_in   = 1 if vn in pits_map else 0
                pit_out  = 1 if (vn-1) in pits_map else 0
 
                if pit_out and (vn-1) in pits_map:
                    _, comp = pits_map[vn-1]
                    idade_pneu = 0
 
                t_ms = gerar_tempo_volta(base_ms,comp,idade_pneu,combustivel,hab,sc_ativo)
                if pit_in:
                    t_ms += rnd_int(18000,28000)
 
                combustivel = max(combustivel - rnd(1.5,2.2), 0)
                idade_pneu += 1
                valida = 0 if (sc_ativo or pit_in or pit_out) else 1
 
                c.execute(
                    "INSERT INTO voltas "
                    "(id_sessao,id_piloto,id_veiculo,numero_volta,tempo_ms,valida,"
                    "pit_in,pit_out,composto_pneu,idade_pneu_voltas,"
                    "combustivel_kg,posicao_inicio,posicao_fim) "
                    "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                    (id_sessao,pid,id_v,vn,t_ms,valida,
                     pit_in,pit_out,comp,idade_pneu,
                     round(combustivel,2),pos_atual,pos_atual))
                id_vv = c.lastrowid
 
                # setores (3 por volta)
                for sn,frac in enumerate([0.30,0.38,0.32], 1):
                    c.execute(
                        "INSERT INTO setores_volta "
                        "(id_volta,numero_setor,tempo_ms,velocidade_max_kmh) "
                        "VALUES (%s,%s,%s,%s)",
                        (id_vv, sn,
                         int(t_ms*frac + rnd_int(-500,500)),
                         rnd(200,360)))
 
                ci_out = pits_map[vn][0] if pit_in else None
                co_out = pits_map[vn][1] if pit_in else None
                voltas_info.append((id_vv, t_ms, pit_in, ci_out, co_out))
 
            id_voltas_piloto[pid] = voltas_info
 
            # pit stops
            for id_vv,_,pit_in,ci,co in voltas_info:
                if pit_in and ci:
                    c.execute(
                        "INSERT INTO pit_stops "
                        "(id_volta,duracao_ms,troca_pneu,composto_entrada,composto_saida) "
                        "VALUES (%s,%s,1,%s,%s)",
                        (id_vv, rnd_int(19000,27000), ci, co))
 
            t_total = sum(t for _,t,*_ in voltas_info)
            resultados_temp.append(
                (t_total, pos_larg, pid, len(voltas_info), pid in abandonam))
 
        # ── RESULTADOS ──────────────────────────────────────────────────────
        resultados_temp.sort(key=lambda x: (x[4], -x[3], x[0]))
        lider_t = resultados_temp[0][0] if resultados_temp else 1
 
        for pos_fin,(t_tot,pos_larg,pid,completou,abnd) in enumerate(resultados_temp, 1):
            id_v = get_veic(pid)
            if not id_v:
                continue
            status = "abandonou" if abnd else "finalizado"
            motivo = random.choice([
                "Problema mecanico","Acidente","Falha eletrica",
                "Pressao oleo","Sobreaquecimento"
            ]) if abnd else None
            pts = PONTOS_F1[pos_fin-1] if pos_fin<=10 and not abnd else 0
 
            c.execute(
                "INSERT INTO resultados "
                "(id_sessao,id_piloto,id_veiculo,posicao_largada,posicao_final,"
                "voltas_completadas,tempo_total_ms,diferenca_lider_ms,"
                "pontos,volta_rapida,status_corrida,motivo_abandono) "
                "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,0,%s,%s)",
                (id_sessao,pid,id_v,pos_larg,pos_fin,
                 completou,t_tot,t_tot-lider_t,pts,status,motivo))
 
            pontos_acc[pid] = pontos_acc.get(pid,0) + pts
            if pos_fin==1 and not abnd:
                stats[pid]["vitorias"] += 1
            if pos_fin<=3 and not abnd:
                stats[pid]["podios"]   += 1
 
        # ── GPS POSICOES (top 3 finais, ate 8 voltas cada) ──────────────────
        top3 = [pid for _,_,pid,_,abnd in resultados_temp[:3] if not abnd]
        for pid in top3:
            id_v = get_veic(pid)
            if not id_v:
                continue
            voltas_info = id_voltas_piloto.get(pid, [])[:8]
            ts_ms = 0
            for id_vv,t_ms,*_ in voltas_info:
                n_pts = 30  # ~30 pontos GPS por volta
                for i in range(n_pts):
                    angulo = 2 * 3.14159 * i / n_pts
                    lat_gps = circ_lat + 0.005 * round(angulo, 4)
                    lon_gps = circ_lon + 0.008 * round(angulo, 4)
                    c.execute(
                        "INSERT INTO gps_posicoes "
                        "(id_sessao,id_veiculo,timestamp_ms,latitude,longitude,"
                        "altitude_m,velocidade_kmh,direcao_graus,precisao_m) "
                        "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                        (id_sessao, id_v,
                         ts_ms + i*(t_ms//n_pts),
                         round(lat_gps,7), round(lon_gps,7),
                         rnd(0,10), rnd(60,360), rnd(0,360), rnd(0.5,3)))
                ts_ms += t_ms
 
        # ── TELEMETRIA BRUTA + RESUMO (vencedor, 5 voltas, 6 sensores) ──────
        if fazer_telemetria and resultados_temp:
            venc_pid  = resultados_temp[0][2]
            id_v_venc = get_veic(venc_pid)
            if id_v_venc and id_v_venc in sensor_ids_veiculo:
                sens_ids    = sensor_ids_veiculo[id_v_venc][:6]
                voltas_venc = id_voltas_piloto.get(venc_pid,[])[:5]
 
                tel_rows = []
                ts_ms = 0
                for id_vv,t_ms,*_ in voltas_venc:
                    for tick in range(0, 90000, 2000):
                        for sid in sens_ids:
                            tel_rows.append((
                                id_sessao, id_vv, id_v_venc, venc_pid, sid,
                                ts_ms+tick, round(random.uniform(100,15000),4), 100))
                    ts_ms += 90000
 
                if tel_rows:
                    c.executemany(
                        "INSERT INTO telemetria "
                        "(id_sessao,id_volta,id_veiculo,id_piloto,id_sensor,"
                        "timestamp_ms,valor,qualidade) "
                        "VALUES (%s,%s,%s,%s,%s,%s,%s,%s)", tel_rows)
 
                # telemetria_volta_resumo
                for id_vv,t_ms,*_ in voltas_venc:
                    c.execute(
                        "INSERT IGNORE INTO telemetria_volta_resumo "
                        "(id_volta,"
                        "rpm_medio,rpm_max,"
                        "temperatura_motor_media,temperatura_motor_max,"
                        "velocidade_media_kmh,velocidade_max_kmh,"
                        "temperatura_freio_dd_media,temperatura_freio_de_media,"
                        "temperatura_freio_td_media,temperatura_freio_te_media,"
                        "pressao_freio_max_bar,"
                        "temperatura_pneu_dd_media,temperatura_pneu_de_media,"
                        "temperatura_pneu_td_media,temperatura_pneu_te_media,"
                        "g_longitudinal_max,g_lateral_max,"
                        "consumo_combustivel_kg,"
                        "energia_recuperada_kj,energia_implantada_kj,"
                        "angulo_volante_medio) "
                        "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                        (id_vv,
                         rnd(10000,14000), rnd(14000,15500),
                         rnd(85,120),      rnd(130,190),
                         rnd(180,280),     rnd(320,380),
                         rnd(400,800),     rnd(400,800),
                         rnd(300,600),     rnd(300,600),
                         rnd(50,150,3),
                         rnd(70,110),      rnd(70,110),
                         rnd(80,120),      rnd(80,120),
                         rnd(3,6,3),       rnd(3,5,3),
                         rnd(1.5,2.5,3),
                         rnd(200,500,3),   rnd(150,400,3),
                         rnd(-45,45,3)))
 
        # ── ALERTAS DE PISTA ────────────────────────────────────────────────
        if sc_volta:
            ts0 = sc_volta * base_ms
            ts1 = (sc_volta + sc_dur) * base_ms
            c.execute(
                "INSERT INTO alertas_pista "
                "(id_sessao,tipo,timestamp_inicio_ms,timestamp_fim_ms,motivo) "
                "VALUES (%s,'safety_car',%s,%s,'Incidente em pista')",
                (id_sessao, ts0, ts1))
            c.execute(
                "INSERT INTO alertas_pista "
                "(id_sessao,tipo,timestamp_inicio_ms,timestamp_fim_ms,motivo) "
                "VALUES (%s,'amarelo',%s,%s,'Detritos na pista')",
                (id_sessao, ts0-5000, ts0))
 
        c.execute(
            "INSERT INTO alertas_pista "
            "(id_sessao,tipo,timestamp_inicio_ms,motivo) "
            "VALUES (%s,'drs_habilitado',%s,'DRS liberado apos volta de formacao')",
            (id_sessao, 200000))
 
        # ── INCIDENTES ──────────────────────────────────────────────────────
        tipos_inc = ["colisao","toque","ultrapassagem_ilegal","ignorar_bandeira",
                     "excesso_velocidade_pit","parametro_tecnico","outro"]
        pens_inc  = ["nenhuma","advertencia","5_segundos","10_segundos",
                     "drive_through","stop_and_go"]
        for _ in range(rnd_int(1, 4)):
            _,pid_i = random.choice(pilotos_lista)
            c.execute(
                "INSERT INTO incidentes "
                "(id_sessao,id_piloto,tipo,descricao,penalidade,volta_ocorrencia) "
                "VALUES (%s,%s,%s,%s,%s,%s)",
                (id_sessao, pid_i,
                 random.choice(tipos_inc),
                 fake.sentence(nb_words=7),
                 random.choice(pens_inc),
                 rnd_int(1, total_voltas)))
 
        # ── FALHAS ──────────────────────────────────────────────────────────
        sistemas = ["motor","freios","suspensao","eletrico",
                    "cambio","ERS","refrigeracao"]
        descrs   = [
            "Temperatura acima do limite",
            "Pressao fora do range operacional",
            "Leitura intermitente do sensor",
            "Vibracao anormal detectada",
            "Queda de tensao na bateria ERS",
            "Vazamento detectado no oleo",
            "Falha de comunicacao CAN bus",
        ]
        for _ in range(rnd_int(0, 4)):
            _,pid_f = random.choice(pilotos_lista)
            eq  = piloto_eq.get(pid_f)
            vs  = veic_por_eq.get(eq, [])
            id_vf = vs[0] if vs else None
            if id_vf:
                c.execute(
                    "INSERT INTO falhas "
                    "(id_sessao,id_veiculo,timestamp_ms,tipo,sistema,descricao,"
                    "valor_lido,valor_esperado,resolvida) "
                    "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                    (id_sessao, id_vf,
                     rnd_int(500000,5000000),
                     random.choice(["critica","alta","media","baixa"]),
                     random.choice(sistemas),
                     random.choice(descrs),
                     rnd(0,200), rnd(0,150), rnd_int(0,1)))
 
        # ── RELATORIOS DE ENGENHARIA ─────────────────────────────────────────
        amostra = random.sample(pilotos_lista, min(3, len(pilotos_lista)))
        for _,(cod,pid) in enumerate(amostra):
            eng = random.choice(ENGENHEIROS)
            c.execute(
                "INSERT INTO relatorios_engenharia "
                "(id_sessao,id_piloto,engenheiro,tipo,titulo,conteudo,versao) "
                "VALUES (%s,%s,%s,%s,%s,%s,1)",
                (id_sessao, pid, eng, "corrida",
                 f"Analise de Corrida GP {apelido} - {cod}",
                 (f"RESUMO: Piloto {cod} completou {total_voltas} voltas. "
                  f"Estrategia de pneus executada conforme planejado. "
                  f"Temperatura media do motor dentro dos parametros. "
                  f"Consumo medio: {rnd(1.5,2.2)} kg/volta. "
                  f"Notas adicionais: {fake.sentence(nb_words=10)}")))
 
    return pontos_acc, stats
 
 
# =============================================================================
#  STEP 8 - USO DE COMPONENTES
# =============================================================================
def step_uso_componentes(c, temp_ids, pilot_ids):
    print("  -> uso_componentes")
    c.execute("SELECT id_componente FROM componentes")
    comp_ids = [r[0] for r in c.fetchall()]
 
    for t_idx,id_temp in enumerate(temp_ids):
        for cod,pid in pilot_ids.items():
            for id_comp in comp_ids:
                ns   = f"COMP-{pid:03d}-{id_comp:02d}-{rnd_int(100,999)}"
                km   = rnd(500,4500)
                hs   = rnd(1,10)
                pen  = 1 if km > 3800 else 0
                ano  = 2024 + t_idx
                c.execute(
                    "INSERT IGNORE INTO uso_componentes "
                    "(id_piloto,id_temporada,id_componente,numero_serie,"
                    "data_introducao,km_acumulado,horas_acumuladas,"
                    "penalidade_grid,status) "
                    "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                    (pid, id_temp, id_comp, ns,
                     f"{ano}-03-01",
                     round(km,2), round(hs,2), pen,
                     random.choice(["em_uso","reserva","descartado"])))
 
 
# =============================================================================
#  STEP 9 - CLASSIFICACAO DO CAMPEONATO
# =============================================================================
def step_classificacao(c, temp_ids, pilot_ids, piloto_eq, pontos_acc, stats):
    print("  -> classificacao_campeonato")
    ranking = sorted(pontos_acc.items(), key=lambda x: -x[1])
    for id_temp in temp_ids:
        for pos,(pid,pts) in enumerate(ranking, 1):
            eq = piloto_eq.get(pid)
            if not eq:
                continue
            st = stats.get(pid, {})
            c.execute(
                "INSERT IGNORE INTO classificacao_campeonato "
                "(id_temporada,id_piloto,id_equipe,posicao,pontos_total,"
                "vitorias,poles,podios,voltas_rapidas) "
                "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                (id_temp, pid, eq, pos, round(pts,2),
                 st.get("vitorias",0), st.get("poles",0),
                 st.get("podios",0),   st.get("vr",0)))
 
 
# =============================================================================
#  STEP 10 - MANUTENCOES
# =============================================================================
def step_manutencoes(c, veic_por_eq):
    print("  -> manutencoes")
    tipos = ["preventiva","corretiva","revisao_pos_evento",
             "troca_componente","upgrade","inspecao"]
    todos = [v for vs in veic_por_eq.values() for v in vs]
 
    for id_v in todos:
        data_b = datetime(2026,3,1)
        for i in range(4):
            d0 = data_b + timedelta(days=14*i + rnd_int(0,3))
            d1 = d0 + timedelta(hours=rnd_int(4,48))
            ce = rnd(5000,150000)
            cr = ce * rnd(0.85,1.2)
            c.execute(
                "INSERT INTO manutencoes "
                "(id_veiculo,tipo,descricao,mecanico_responsavel,"
                "data_inicio,data_fim,custo_estimado,custo_real,status) "
                "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,'concluida')",
                (id_v, random.choice(tipos),
                 fake.sentence(nb_words=8),
                 random.choice(MECANICOS),
                 d0.strftime("%Y-%m-%d %H:%M:%S"),
                 d1.strftime("%Y-%m-%d %H:%M:%S"),
                 round(ce,2), round(cr,2)))

In [7]:
# main
def main():
    print("=" * 64)
    print("  APEX MOTORSPORT - Populador Completo (32 tabelas)")
    print(f"  Volume: {VOLUME.upper()}")
    print("=" * 64)
 
    conn = connect()
    c    = new_cur(conn)
 
    try:
        print("\n[1/10] Referencias basicas...")
        step_referencias(c)
        conn.commit()
 
        print("\n[2/10] Fabricantes, motores, equipes, pilotos...")
        eq_ids, pilot_ids, fab_ids = step_equipes_pilotos(c)
        conn.commit()
 
        print("\n[3/10] Circuitos...")
        circ_ids = step_circuitos(c)
        conn.commit()
 
        print("\n[4/10] Veiculos e modelos...")
        veic_por_eq = step_veiculos(c, eq_ids, fab_ids)
        conn.commit()
 
        print("\n[5/10] Sensores...")
        sensor_ids_veiculo = step_sensores(c, veic_por_eq)
        conn.commit()
 
        print("\n[6/10] Temporadas, eventos, sessoes, meteorologia...")
        temp_ids, eventos_info = step_temporadas_eventos(
            c, circ_ids, CFG["temporadas"], CFG["eventos_por_temp"])
        conn.commit()
 
        print("\n[7/10] Corridas completas...")
        c.execute("SELECT id_piloto,id_equipe FROM pilotos")
        piloto_eq = {r[0]:r[1] for r in c.fetchall()}
 
        pontos, stats = step_corridas(
            c, eventos_info, pilot_ids, eq_ids, veic_por_eq,
            sensor_ids_veiculo, CFG["voltas_corrida"], CFG["telemetria"])
        conn.commit()
 
        print("\n[8/10] Uso de componentes...")
        step_uso_componentes(c, temp_ids, pilot_ids)
        conn.commit()
 
        print("\n[9/10] Classificacao do campeonato...")
        step_classificacao(c, temp_ids, pilot_ids, piloto_eq, pontos, stats)
        conn.commit()
 
        print("\n[10/10] Manutencoes...")
        step_manutencoes(c, veic_por_eq)
        conn.commit()
 
        # ── RESUMO FINAL ─────────────────────────────────────────────────────
        print("\n" + "=" * 64)
        print("  CONCLUIDO! Registros por tabela:")
        print("=" * 64)
 
        todas = [
            "alertas_pista","canais_telemetria","categorias","circuitos",
            "classificacao_campeonato","componentes","equipes","eventos",
            "fabricantes","falhas","gps_posicoes","incidentes","manutencoes",
            "meteorologia","modelos_veiculo","motores","paises","pilotos",
            "pit_stops","relatorios_engenharia","resultados","sensores",
            "sessoes","setores_volta","setups_veiculo","telemetria",
            "telemetria_volta_resumo","temporadas","tipos_sensor",
            "uso_componentes","veiculos","voltas",
        ]
        total = 0
        for t in todas:
            c.execute(f"SELECT COUNT(*) FROM {t}")
            n = c.fetchone()[0]
            total += n
            status = "OK" if n > 0 else "!!"
            print(f"  [{status}] {t:<35} {n:>8} registros")
 
        print(f"\n  TOTAL GERAL: {total:,} registros em 32 tabelas")
        print("=" * 64)
        print("\n  Banco pronto para o Power BI! 🏁\n")
 
    except Exception as e:
        conn.rollback()
        print(f"\n ERRO: {e}")
        import traceback; traceback.print_exc()
    finally:
        c.close()
        conn.close()
 
 
if __name__ == "__main__":
    main()

  APEX MOTORSPORT - Populador Completo (32 tabelas)
  Volume: MEDIO

[1/10] Referencias basicas...
  -> paises, categorias, tipos_sensor, componentes, canais_telemetria

[2/10] Fabricantes, motores, equipes, pilotos...
  -> fabricantes, motores, equipes, pilotos

[3/10] Circuitos...
  -> circuitos

[4/10] Veiculos e modelos...
  -> modelos_veiculo, veiculos

[5/10] Sensores...
  -> sensores

[6/10] Temporadas, eventos, sessoes, meteorologia...
  -> temporadas, eventos, sessoes, meteorologia

[7/10] Corridas completas...
  -> corridas: 20 eventos x 60 voltas x 20 pilotos
     [01/20] Montreal (seco)

 ERRO: 1264 (22003): Out of range value for column 'mola_dianteira_nm' at row 1


Traceback (most recent call last):
  File "c:\Users\rafae\AppData\Local\Programs\Python\Python314\Lib\site-packages\mysql\connector\connection_cext.py", line 772, in cmd_query
    self._cmysql.query(
    ~~~~~~~~~~~~~~~~~~^
        query,
        ^^^^^^
    ...<3 lines>...
        query_attrs=self.query_attrs,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
_mysql_connector.MySQLInterfaceError: Out of range value for column 'mola_dianteira_nm' at row 1

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\rafae\AppData\Local\Temp\ipykernel_15524\345089058.py", line 41, in main
    pontos, stats = step_corridas(
                    ~~~~~~~~~~~~~^
        c, eventos_info, pilot_ids, eq_ids, veic_por_eq,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        sensor_ids_veiculo, CFG["voltas_corrida"], CFG["telemetria"])
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\rafa

In [4]:
"""
=============================================================================
 APEX MOTORSPORT - POPULADOR COMPLETO DO BANCO DE DADOS
 Popula TODAS as 32 tabelas com dados realistas

 INSTALACAO (rode uma vez no terminal):
   pip install mysql-connector-python faker

 CONFIGURACAO:
   Ajuste DB_CONFIG abaixo (user/password)

 EXECUCAO:
   python popular_banco.py
=============================================================================
"""

import random
from datetime import datetime, timedelta, date
from faker import Faker
import mysql.connector

fake = Faker("pt_BR")
random.seed(42)

# =============================================================================
#  CONFIGURACOES
# =============================================================================
DB_CONFIG = {
    "host":     "127.0.0.1",
    "port":     3306,
    "user":     "root",    # <-- ajuste
    "password": "root",        # <-- ajuste
    "database": "apex_telemetria",
    "charset":  "utf8mb4",
}

VOLUME = "medio"  # "pequeno" | "medio" | "grande"

VOLUMES = {
    "pequeno": {"temporadas": 1, "eventos_por_temp": 10, "voltas_corrida": 40, "telemetria": False},
    "medio":   {"temporadas": 2, "eventos_por_temp": 15, "voltas_corrida": 60, "telemetria": True},
    "grande":  {"temporadas": 3, "eventos_por_temp": 22, "voltas_corrida": random.randint(49, 78), "telemetria": True},
}
CFG = VOLUMES[VOLUME]

# =============================================================================
#  HELPERS
# =============================================================================
def connect():
    return mysql.connector.connect(**DB_CONFIG)

def new_cur(conn):
    """Cursor com buffered=True evita 'Unread result found'."""
    return conn.cursor(buffered=True)

def rnd(a, b, dec=2):
    return round(random.uniform(a, b), dec)

def rnd_int(a, b):
    return random.randint(a, b)

# =============================================================================
#  DADOS MESTRES
# =============================================================================
PAISES = [
    ("BRA","Brasil","America do Sul"),   ("GBR","Reino Unido","Europa"),
    ("ITA","Italia","Europa"),           ("AUT","Austria","Europa"),
    ("DEU","Alemanha","Europa"),         ("NLD","Paises Baixos","Europa"),
    ("MCO","Monaco","Europa"),           ("ESP","Espanha","Europa"),
    ("FRA","Franca","Europa"),           ("USA","Estados Unidos","America do Norte"),
    ("AUS","Australia","Oceania"),       ("JPN","Japao","Asia"),
    ("MEX","Mexico","America do Norte"), ("SGP","Singapura","Asia"),
    ("ARE","Emirados Arabes","Asia"),    ("BEL","Belgica","Europa"),
    ("HUN","Hungria","Europa"),          ("BAH","Bahrein","Asia"),
    ("CAN","Canada","America do Norte"), ("CHN","China","Asia"),
    ("FIN","Finlandia","Europa"),        ("DNK","Dinamarca","Europa"),
    ("NZL","Nova Zelandia","Oceania"),
]

EQUIPES_DATA = [
    ("Mercedes-AMG PETRONAS",       "Mercedes",    "GBR", "Toto Wolff",       2010),
    ("Scuderia Ferrari",            "Ferrari",     "ITA", "Fred Vasseur",     1929),
    ("Oracle Red Bull Racing",      "Red Bull",    "AUT", "Christian Horner", 2005),
    ("McLaren F1 Team",             "McLaren",     "GBR", "Andrea Stella",    1963),
    ("Aston Martin Aramco F1 Team", "Aston Martin","GBR", "Mike Krack",       2021),
    ("BWT Alpine F1 Team",          "Alpine",      "FRA", "Oliver Oakes",     1977),
    ("Williams Racing",             "Williams",    "GBR", "James Vowles",     1977),
    ("MoneyGram Haas F1 Team",      "Haas",        "USA", "Ayao Komatsu",     2016),
    ("Visa Cash App RB F1 Team",    "Racing Bulls","ITA", "Laurent Mekies",   2006),
    ("Stake F1 Team Kick Sauber",   "Sauber",      "DEU", "Mattia Binotto",   1993),
]

PILOTOS_DATA = [
    # (codigo, nome, sobrenome, iso_pais, nascimento, numero, equipe_idx)
    ("HAM","Lewis",    "Hamilton",  "GBR","1985-01-07", 44, 1),
    ("RUS","George",   "Russell",   "GBR","1998-02-15", 63, 0),
    ("VER","Max",      "Verstappen","NLD","1997-09-30",  1, 2),
    ("NOR","Lando",    "Norris",    "GBR","1999-11-13",  4, 3),
    ("LEC","Charles",  "Leclerc",   "MCO","1997-10-16", 16, 1),
    ("PIA","Oscar",    "Piastri",   "AUS","2001-04-06", 81, 3),
    ("ALO","Fernando", "Alonso",    "ESP","1981-07-29", 14, 4),
    ("STR","Lance",    "Stroll",    "CAN","1998-10-29", 18, 4),
    ("GAS","Pierre",   "Gasly",     "FRA","1996-02-07", 10, 5),
    ("OCO","Esteban",  "Ocon",      "FRA","1996-09-17", 31, 5),
    ("ALB","Alexander","Albon",     "GBR","1996-03-23", 23, 6),
    ("SAR","Logan",    "Sargeant",  "USA","2000-12-31",  2, 6),
    ("MAG","Kevin",    "Magnussen", "DNK","1992-10-05", 20, 7),
    ("HUL","Nico",     "Hulkenberg","DEU","1987-08-19", 27, 7),
    ("TSU","Yuki",     "Tsunoda",   "JPN","2000-05-11", 22, 8),
    ("LAW","Liam",     "Lawson",    "NZL","2002-02-11", 30, 8),
    ("BOT","Valtteri", "Bottas",    "FIN","1989-08-28", 77, 9),
    ("ZHO","Guanyu",   "Zhou",      "CHN","1999-05-30", 24, 9),
    ("ANT","Andrea",   "Antonelli", "ITA","2006-08-25", 12, 0),
    ("BEA","Oliver",   "Bearman",   "GBR","2005-05-08", 87, 1),
]

CIRCUITOS_DATA = [
    # nome, apelido, iso, cidade, comp_m, curvas, sentido, alt, lat, lon
    ("Autodromo Jose Carlos Pace",    "Interlagos","BRA","Sao Paulo",       4309,15,"anti-horário", 785,-23.7036,-46.6975),
    ("Circuit de Monaco",             "Monaco",    "MCO","Monte Carlo",     3337,19,"horário",        7, 43.7347,  7.4205),
    ("Silverstone Circuit",           "Silverstone","GBR","Silverstone",    5891,18,"horário",      153, 52.0786, -1.0169),
    ("Autodromo Nazionale Monza",     "Monza",     "ITA","Monza",           5793,11,"horário",      162, 45.6156,  9.2811),
    ("Circuit de Barcelona-Catalunya","Barcelona", "ESP","Barcelona",       4657,16,"horário",      115, 41.5700,  2.2611),
    ("Hungaroring",                   "Budapest",  "HUN","Budapest",        4381,14,"horário",      264, 47.5789, 19.2486),
    ("Circuit Gilles Villeneuve",     "Montreal",  "CAN","Montreal",        4361,14,"anti-horário",  13, 45.5000,-73.5226),
    ("Suzuka Circuit",                "Suzuka",    "JPN","Suzuka",          5807,18,"misto",          40, 34.8431,136.5411),
    ("Marina Bay Street Circuit",     "Singapura", "SGP","Singapura",       4940,23,"horário",        15,  1.2914,103.8638),
    ("Yas Marina Circuit",            "Abu Dhabi", "ARE","Abu Dhabi",       5281,16,"anti-horário",    3, 24.4672, 54.6031),
    ("Spa-Francorchamps",             "Spa",       "BEL","Spa",             7004,20,"horário",       401, 50.4372,  5.9715),
    ("Bahrain International Circuit", "Bahrein",   "BAH","Sakhir",          5412,15,"horário",         7, 26.0325, 50.5106),
    ("Shanghai International Circuit","Xangai",    "CHN","Xangai",          5451,16,"horário",         5, 31.3389,121.2198),
    ("Circuit of The Americas",       "Austin",    "USA","Austin",          5513,20,"anti-horário",  161, 30.1328,-97.6411),
    ("Autodromo Hermanos Rodriguez",  "Mexico",    "MEX","Cidade do Mexico",4304,17,"horário",      2240, 19.4042,-99.0907),
]

TIPOS_SENSOR_DATA = [
    ("RPM",          "Rotacao Motor",             "motor",       "rpm",    0,  20000, 200),
    ("TEMP_MOT",     "Temperatura Motor",         "motor",       "C",      0,    200,  10),
    ("PRESS_OLEO",   "Pressao do Oleo",           "motor",       "bar",    0,     15,  50),
    ("CONSUMO",      "Consumo Combustivel",       "motor",       "kg/lap", 0,      5,  10),
    ("COMBUSTIVEL",  "Nivel Combustivel",         "motor",       "kg",     0,    110,   5),
    ("VELOC",        "Velocidade Linear",         "aerodinamica","km/h",   0,    400,  50),
    ("ACCEL_LONG",   "Aceleracao Longitudinal",   "suspensao",   "g",     -8,      8, 200),
    ("ACCEL_LAT",    "Aceleracao Lateral",        "suspensao",   "g",     -7,      7, 200),
    ("TEMP_PNEU_DD", "Temp Pneu Diant Dir",       "pneu",        "C",      0,    150,  10),
    ("TEMP_PNEU_DE", "Temp Pneu Diant Esq",       "pneu",        "C",      0,    150,  10),
    ("TEMP_PNEU_TD", "Temp Pneu Tras Dir",        "pneu",        "C",      0,    150,  10),
    ("TEMP_PNEU_TE", "Temp Pneu Tras Esq",        "pneu",        "C",      0,    150,  10),
    ("PRESS_PNEU_DD","Pressao Pneu Diant Dir",    "pneu",        "psi",   15,     35,  10),
    ("PRESS_PNEU_DE","Pressao Pneu Diant Esq",    "pneu",        "psi",   15,     35,  10),
    ("PRESS_PNEU_TD","Pressao Pneu Tras Dir",     "pneu",        "psi",   15,     35,  10),
    ("PRESS_PNEU_TE","Pressao Pneu Tras Esq",     "pneu",        "psi",   15,     35,  10),
    ("TEMP_FREIO_DD","Temp Freio Diant Dir",      "freio",       "C",      0,   1200,  20),
    ("TEMP_FREIO_DE","Temp Freio Diant Esq",      "freio",       "C",      0,   1200,  20),
    ("TEMP_FREIO_TD","Temp Freio Tras Dir",       "freio",       "C",      0,   1200,  20),
    ("TEMP_FREIO_TE","Temp Freio Tras Esq",       "freio",       "C",      0,   1200,  20),
    ("PRESS_FREIO",  "Pressao Linha Freio",       "freio",       "bar",    0,    200, 100),
    ("VOLT_ERS",     "Tensao Bateria ERS",        "eletrico",    "V",      0,   1000,  10),
    ("CARGA_ERS",    "Carga Bateria ERS",         "eletrico",    "pct",    0,    100,  10),
    ("ACELERADOR",   "Posicao Acelerador",        "motor",       "pct",    0,    100, 100),
    ("FREIO_PEDAL",  "Posicao Freio",             "freio",       "pct",    0,    100, 100),
    ("MARCHA",       "Marcha Engatada",           "transmissao", "n",     -1,      8,  50),
    ("DRS",          "Status DRS",               "aerodinamica","bool",   0,      1,  10),
    ("GPS_LAT",      "Latitude GPS",             "gps",         "graus", -90,    90,  50),
    ("GPS_LON",      "Longitude GPS",            "gps",         "graus",-180,   180,  50),
    ("FREQ_CARD",    "Frequencia Cardiaca",       "piloto",      "bpm",   40,    220,   1),
    ("TEMP_COCKPIT", "Temperatura Cockpit",       "piloto",      "C",      0,     60,   5),
]

PONTOS_F1   = [25,18,15,12,10,8,6,4,2,1]
COMPOSTOS_S = ["slick_soft","slick_medium","slick_hard"]

TEMPO_BASE = {
    "Interlagos":73000,"Monaco":72000,"Silverstone":85000,"Monza":80000,
    "Barcelona":75000,"Budapest":76000,"Montreal":73000,"Suzuka":89000,
    "Singapura":95000,"Abu Dhabi":84000,"Spa":103000,"Bahrein":90000,
    "Xangai":93000,"Austin":94000,"Mexico":78000,
}

HABILIDADE = {
    "VER":0,"NOR":200,"HAM":100,"LEC":150,"PIA":300,"RUS":350,
    "ALO":400,"BEA":450,"GAS":500,"ALB":600,"STR":700,"HUL":650,
    "TSU":750,"LAW":800,"BOT":900,"MAG":950,"ZHO":1000,"OCO":850,
    "ANT":1100,"SAR":1200,
}

ENGENHEIROS = [
    "Peter Bonnington","Riccardo Adami","Gianpiero Lambiase","Will Joseph",
    "Tom Stallard","Chris Cronin","Gaetan Jego","Francesco Nenci",
    "Andrew Shovlin","Xevi Pujolar","Alan Permane","Bradley Joyce",
]

MECANICOS = [
    "Joao Silva","Carlos Mendes","Hans Muller","Luigi Ferrari",
    "James Cooper","Pierre Dubois","Mario Rossi","Ahmed Al-Rashid",
    "Pedro Costa","Ralf Schmidt","Marco Bianchi","Steve Thompson",
]

# =============================================================================
#  FUNCOES DE GERACAO
# =============================================================================
def gerar_tempo_volta(base_ms, composto, idade_pneu, combustivel_kg, habilidade=0, sc=False):
    t = base_ms + habilidade
    t += {"slick_soft":-400,"slick_medium":0,"slick_hard":300,
          "intermediario":3000,"chuva":7000}.get(composto, 0)
    t += {"slick_soft":35,"slick_medium":18,"slick_hard":8,
          "intermediario":25,"chuva":40}.get(composto, 15) * idade_pneu
    t += combustivel_kg * 30
    t += rnd_int(-200, 200)
    if sc:
        t += rnd_int(15000, 25000)
    return max(int(t), base_ms - 2000)


def estrategia_pit(total_voltas, condicao):
    meio = total_voltas // 2
    if condicao in ("molhado","muito_molhado"):
        return random.choice([
            [(12,"chuva","intermediario"),(36,"intermediario","slick_medium")],
            [(18,"chuva","intermediario")],
        ])
    return random.choice([
        [(meio + rnd_int(-3,3), "slick_medium","slick_hard")],
        [(meio - 10,            "slick_soft",  "slick_hard")],
        [(total_voltas//3,      "slick_soft",  "slick_medium"),
         (total_voltas*2//3,    "slick_medium","slick_hard")],
    ])


# =============================================================================
#  STEP 1 - REFERENCIAS BASICAS
# =============================================================================
def step_referencias(c):
    print("  -> paises, categorias, tipos_sensor, componentes, canais_telemetria")

    c.executemany(
        "INSERT IGNORE INTO paises (codigo_iso,nome,continente) VALUES (%s,%s,%s)",
        PAISES)

    c.execute(
        "INSERT IGNORE INTO categorias (nome,descricao) VALUES (%s,%s)",
        ("Formula 1","Campeonato Mundial FIA"))

    for row in TIPOS_SENSOR_DATA:
        c.execute(
            "INSERT IGNORE INTO tipos_sensor "
            "(codigo,nome,categoria,unidade,valor_min,valor_max,frequencia_hz) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s)", row)

    comps = [
        ("Motor de Combustao Interna","motor",         4000, None),
        ("Caixa de Cambio",           "caixa_cambio",  None,  6.0),
        ("MGU-H",                     "motor_energia",  None, None),
        ("MGU-K",                     "motor_energia",  None, None),
        ("Bateria ERS",               "baterias",       None, None),
        ("Turbocompressor",           "turbo",          None, None),
        ("Unidade Eletronica ECU",    "unidade_eletrica",None,None),
    ]
    c.executemany(
        "INSERT IGNORE INTO componentes "
        "(nome,tipo,vida_util_km,vida_util_horas) VALUES (%s,%s,%s,%s)", comps)

    # canais_telemetria
    c.execute("SELECT id_tipo_sensor, codigo, unidade FROM tipos_sensor")
    tipos = c.fetchall()
    grupos = {
        "RPM":"Motor","TEMP_MOT":"Motor","PRESS_OLEO":"Motor",
        "CONSUMO":"Motor","COMBUSTIVEL":"Motor","ACELERADOR":"Motor",
        "MARCHA":"Transmissao","VELOC":"Aerodinamica","DRS":"Aerodinamica",
        "ACCEL_LONG":"Dinamica","ACCEL_LAT":"Dinamica",
        "TEMP_PNEU_DD":"Pneus","TEMP_PNEU_DE":"Pneus",
        "TEMP_PNEU_TD":"Pneus","TEMP_PNEU_TE":"Pneus",
        "PRESS_PNEU_DD":"Pneus","PRESS_PNEU_DE":"Pneus",
        "PRESS_PNEU_TD":"Pneus","PRESS_PNEU_TE":"Pneus",
        "TEMP_FREIO_DD":"Freios","TEMP_FREIO_DE":"Freios",
        "TEMP_FREIO_TD":"Freios","TEMP_FREIO_TE":"Freios",
        "PRESS_FREIO":"Freios","FREIO_PEDAL":"Freios",
        "VOLT_ERS":"ERS","CARGA_ERS":"ERS",
        "GPS_LAT":"GPS","GPS_LON":"GPS",
        "FREQ_CARD":"Piloto","TEMP_COCKPIT":"Piloto",
    }
    for id_tipo, codigo, unidade in tipos:
        grupo  = grupos.get(codigo, "Geral")
        formula = f"AVG({codigo})" if ("TEMP" in codigo or "PRESS" in codigo) else f"MAX({codigo})"
        c.execute(
            "INSERT IGNORE INTO canais_telemetria "
            "(nome,grupo,id_tipo_sensor,formula_calculo,unidade,descricao) "
            "VALUES (%s,%s,%s,%s,%s,%s)",
            (f"Canal_{codigo}", grupo, id_tipo, formula, unidade,
             f"Canal de analise para o sensor {codigo}"))


# =============================================================================
#  STEP 2 - FABRICANTES, MOTORES, EQUIPES, PILOTOS
# =============================================================================
def step_equipes_pilotos(c):
    print("  -> fabricantes, motores, equipes, pilotos")

    fabs = [
        ("Mercedes-AMG HPP",        "GBR"),
        ("Scuderia Ferrari SpA",     "ITA"),
        ("Red Bull Powertrains",     "AUT"),
        ("McLaren Racing",           "GBR"),
        ("Renault Sport",            "FRA"),
        ("Honda Racing Corp",        "JPN"),
        ("Aston Martin Perf Tech",   "GBR"),
    ]
    fab_ids = {}
    for nome, iso in fabs:
        c.execute("SELECT id_pais FROM paises WHERE codigo_iso=%s", (iso,))
        row = c.fetchone()
        id_pais = row[0] if row else 1
        c.execute(
            "INSERT IGNORE INTO fabricantes (nome,id_pais) VALUES (%s,%s)",
            (nome, id_pais))
        c.execute(
            "SELECT id_fabricante FROM fabricantes WHERE nome=%s", (nome,))
        fab_ids[nome] = c.fetchone()[0]

    id_merc = fab_ids["Mercedes-AMG HPP"]
    id_ferr = fab_ids["Scuderia Ferrari SpA"]
    id_rb   = fab_ids["Red Bull Powertrains"]
    id_mcl  = fab_ids["McLaren Racing"]

    motores = [
        (id_merc,"M17-EQ",   "Mercedes M17 E Performance","hibrido",6,1600,1060,None),
        (id_ferr,"F067-26",  "Ferrari 067/26",             "hibrido",6,1600,1040,None),
        (id_rb,  "RBPTH002", "Honda RBPTH002",             "hibrido",6,1600,1020,None),
        (id_mcl, "M17-MCL",  "Mercedes M17 cliente",       "hibrido",6,1600,1050,None),
    ]
    for m in motores:
        c.execute(
            "INSERT IGNORE INTO motores "
            "(id_fabricante,codigo,nome,tipo,cilindros,cilindrada_cc,potencia_cv,torque_nm) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s,%s)", m)

    eq_ids = []
    for nome,apelido,iso,diretor,fund in EQUIPES_DATA:
        c.execute("SELECT id_pais FROM paises WHERE codigo_iso=%s", (iso,))
        row = c.fetchone(); id_pais = row[0] if row else 1
        c.execute(
            "INSERT IGNORE INTO equipes "
            "(nome,apelido,id_pais,diretor,ano_fundacao) VALUES (%s,%s,%s,%s,%s)",
            (nome,apelido,id_pais,diretor,fund))
        c.execute("SELECT id_equipe FROM equipes WHERE nome=%s", (nome,))
        eq_ids.append(c.fetchone()[0])

    pilot_ids = {}
    for cod,nome,sob,iso,nasc,num,eq_idx in PILOTOS_DATA:
        c.execute("SELECT id_pais FROM paises WHERE codigo_iso=%s", (iso,))
        row = c.fetchone(); id_pais = row[0] if row else 1
        c.execute(
            "INSERT IGNORE INTO pilotos "
            "(codigo_piloto,nome,sobrenome,id_pais,data_nascimento,numero_carro,id_equipe) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s)",
            (cod,nome,sob,id_pais,nasc,num,eq_ids[eq_idx]))
        c.execute(
            "SELECT id_piloto FROM pilotos WHERE codigo_piloto=%s", (cod,))
        pilot_ids[cod] = c.fetchone()[0]

    return eq_ids, pilot_ids, fab_ids


# =============================================================================
#  STEP 3 - CIRCUITOS
# =============================================================================
def step_circuitos(c):
    print("  -> circuitos")
    circ_ids = {}
    for nome,apelido,iso,cidade,comp,curv,sent,alt,lat,lon in CIRCUITOS_DATA:
        c.execute("SELECT id_pais FROM paises WHERE codigo_iso=%s", (iso,))
        row = c.fetchone(); id_pais = row[0] if row else 1
        c.execute(
            "INSERT IGNORE INTO circuitos "
            "(nome,apelido,id_pais,cidade,comprimento_m,numero_curvas,"
            "sentido,altitude_m,latitude,longitude) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
            (nome,apelido,id_pais,cidade,comp,curv,sent,alt,lat,lon))
        c.execute(
            "SELECT id_circuito FROM circuitos WHERE apelido=%s", (apelido,))
        circ_ids[apelido] = c.fetchone()[0]
    return circ_ids


# =============================================================================
#  STEP 4 - VEICULOS
# =============================================================================
def step_veiculos(c, eq_ids, fab_ids):
    print("  -> modelos_veiculo, veiculos")

    id_merc = fab_ids["Mercedes-AMG HPP"]
    id_ferr = fab_ids["Scuderia Ferrari SpA"]
    id_rb   = fab_ids["Red Bull Powertrains"]
    id_mcl  = fab_ids["McLaren Racing"]
    id_ren  = fab_ids["Renault Sport"]
    id_am   = fab_ids["Aston Martin Perf Tech"]

    modelos_raw = [
        (id_merc,"W17",2026),(id_ferr,"SF-26",2026),(id_rb,"RB22",2026),
        (id_mcl,"MCL40",2026),(id_ren,"A526",2026),(id_am,"AMR26",2026),
        (id_merc,"FW47",2026),(id_am,"VF-26",2026),(id_rb,"VCARB3",2026),
        (id_ferr,"C46",2026),
    ]
    mod_ids = []
    for id_fab,nome,ano in modelos_raw:
        c.execute(
            "INSERT IGNORE INTO modelos_veiculo "
            "(id_fabricante,nome,ano_modelo,categoria) VALUES (%s,%s,%s,'Formula 1')",
            (id_fab,nome,ano))
        c.execute(
            "SELECT id_modelo FROM modelos_veiculo WHERE nome=%s AND ano_modelo=%s",
            (nome,ano))
        mod_ids.append(c.fetchone()[0])

    siglas  = ["W17","SF26","RB22","MCL","ALP","AMR","FW","VF","VCB","C46"]
    numeros = [(63,44),(16,87),(1,30),(4,81),(10,31),(14,18),(23,2),(27,20),(22,30),(77,24)]

    veic_por_eq = {}
    for i,(eq_id,(n1,n2)) in enumerate(zip(eq_ids, numeros)):
        mid = mod_ids[i]
        veic_por_eq[eq_id] = []
        for j,num in enumerate([n1,n2]):
            chassi = f"{siglas[i]}-2026-{num:03d}-{rnd_int(1000,9999)}"
            try:
                c.execute(
                    "INSERT INTO veiculos "
                    "(id_modelo,id_equipe,numero_chassi,numero_carro,"
                    "data_fabricacao,km_total,status) "
                    "VALUES (%s,%s,%s,%s,'2026-01-15',0,'ativo')",
                    (mid,eq_id,chassi,num))
                veic_por_eq[eq_id].append(c.lastrowid)
            except Exception:
                c.execute(
                    "SELECT id_veiculo FROM veiculos WHERE numero_chassi=%s",
                    (chassi,))
                row = c.fetchone()
                if row:
                    veic_por_eq[eq_id].append(row[0])
    return veic_por_eq


# =============================================================================
#  STEP 5 - SENSORES
# =============================================================================
def step_sensores(c, veic_por_eq):
    print("  -> sensores")
    c.execute("SELECT id_tipo_sensor, codigo FROM tipos_sensor")
    tipos = c.fetchall()

    pos_map = {
        "RPM":"bloco motor","TEMP_MOT":"cabeca cilindro","PRESS_OLEO":"circuito oleo",
        "VELOC":"chassi central","ACCEL_LONG":"chassi central","ACCEL_LAT":"chassi central",
        "TEMP_PNEU_DD":"roda DD","TEMP_PNEU_DE":"roda DE",
        "TEMP_PNEU_TD":"roda TD","TEMP_PNEU_TE":"roda TE",
        "PRESS_PNEU_DD":"roda DD","PRESS_PNEU_DE":"roda DE",
        "PRESS_PNEU_TD":"roda TD","PRESS_PNEU_TE":"roda TE",
        "TEMP_FREIO_DD":"pastilha DD","TEMP_FREIO_DE":"pastilha DE",
        "TEMP_FREIO_TD":"pastilha TD","TEMP_FREIO_TE":"pastilha TE",
        "PRESS_FREIO":"cilindro mestre","VOLT_ERS":"bateria ERS","CARGA_ERS":"bateria ERS",
        "ACELERADOR":"pedal acelerador","FREIO_PEDAL":"pedal freio",
        "MARCHA":"caixa cambio","DRS":"asa traseira",
        "GPS_LAT":"antena GPS","GPS_LON":"antena GPS",
        "FREQ_CARD":"assento piloto","TEMP_COCKPIT":"painel cockpit",
        "CONSUMO":"sistema combustivel","COMBUSTIVEL":"tanque",
    }

    sensor_ids_veiculo = {}
    todos = [v for vs in veic_por_eq.values() for v in vs]

    for id_v in todos:
        sensor_ids_veiculo[id_v] = []
        for id_t, cod in tipos:
            ns  = f"SN-{id_v:03d}-{id_t:03d}-{rnd_int(100,9999)}"
            pos = pos_map.get(cod, "chassi")
            c.execute(
                "INSERT IGNORE INTO sensores "
                "(id_veiculo,id_tipo_sensor,numero_serie,posicao,data_instalacao,ativo) "
                "VALUES (%s,%s,%s,%s,'2026-01-10',1)",
                (id_v,id_t,ns,pos))
            if c.lastrowid:
                sensor_ids_veiculo[id_v].append(c.lastrowid)

        if not sensor_ids_veiculo[id_v]:
            c.execute(
                "SELECT id_sensor FROM sensores WHERE id_veiculo=%s LIMIT 10",
                (id_v,))
            sensor_ids_veiculo[id_v] = [r[0] for r in c.fetchall()]

    return sensor_ids_veiculo


# =============================================================================
#  STEP 6 - TEMPORADAS, EVENTOS, SESSOES, METEOROLOGIA
# =============================================================================
def step_temporadas_eventos(c, circ_ids, n_temp, n_eventos):
    print("  -> temporadas, eventos, sessoes, meteorologia")
    c.execute(
        "SELECT id_categoria FROM categorias WHERE nome='Formula 1'")
    id_cat = c.fetchone()[0]

    temp_ids = []
    for off in range(n_temp):
        ano = 2024 + off
        c.execute(
            "INSERT IGNORE INTO temporadas (id_categoria,ano,descricao) VALUES (%s,%s,%s)",
            (id_cat, ano, f"Campeonato Mundial F1 {ano}"))
        c.execute(
            "SELECT id_temporada FROM temporadas WHERE id_categoria=%s AND ano=%s",
            (id_cat,ano))
        temp_ids.append(c.fetchone()[0])

    circs = list(CIRCUITOS_DATA)
    random.shuffle(circs)
    circs = circs[:n_eventos]

    eventos_info = []  # (id_evento, id_sessao_corrida, apelido, condicao, lat, lon)

    for t_idx, id_temp in enumerate(temp_ids):
        data_ref = date(2024 + t_idx, 3, 1)

        for rodada,(nome_c,apelido,_,_,_,_,_,_,lat,lon) in enumerate(circs, 1):
            id_circ = circ_ids.get(apelido)
            if not id_circ:
                continue

            roll     = random.random()
            condicao = "seco" if roll<0.70 else ("umido" if roll<0.85 else "molhado")
            temp_ar  = rnd(18,35)
            temp_pi  = rnd(25,55)
            umid     = rnd(40,90)
            pres     = rnd(1005,1020)
            vento    = rnd(5,35)

            d_ini = data_ref
            d_fim = data_ref + timedelta(days=2)

            c.execute(
                "INSERT INTO eventos "
                "(id_temporada,id_circuito,nome,rodada,data_inicio,data_fim,status) "
                "VALUES (%s,%s,%s,%s,%s,%s,'concluido')",
                (id_temp, id_circ,
                 f"Grande Premio de {apelido} {2024+t_idx}",
                 rodada,
                 d_ini.strftime("%Y-%m-%d"), d_fim.strftime("%Y-%m-%d")))
            id_ev = c.lastrowid

            sessoes_tipos = [
                ("treino_livre_1", d_ini,              "14:00:00","16:00:00"),
                ("treino_livre_2", d_ini,              "18:00:00","19:00:00"),
                ("treino_livre_3", d_ini+timedelta(1), "13:00:00","14:00:00"),
                ("classificacao",  d_ini+timedelta(1), "16:00:00","17:00:00"),
                ("corrida",        d_fim,              "15:00:00","17:30:00"),
            ]
            id_corrida = None
            for tipo,dia,h0,h1 in sessoes_tipos:
                dt0 = datetime.strptime(f"{dia} {h0}", "%Y-%m-%d %H:%M:%S")
                dt1 = datetime.strptime(f"{dia} {h1}", "%Y-%m-%d %H:%M:%S")
                c.execute(
                    "INSERT INTO sessoes "
                    "(id_evento,tipo,numero,data_hora_inicio,data_hora_fim,"
                    "condicao_pista,temperatura_ar,temperatura_pista,"
                    "umidade_pct,pressao_hpa,velocidade_vento_kmh) "
                    "VALUES (%s,%s,1,%s,%s,%s,%s,%s,%s,%s,%s)",
                    (id_ev, tipo,
                     dt0.strftime("%Y-%m-%d %H:%M:%S"),
                     dt1.strftime("%Y-%m-%d %H:%M:%S"),
                     condicao,temp_ar,temp_pi,umid,pres,vento))
                if tipo == "corrida":
                    id_corrida = c.lastrowid

            # Meteorologia horaria no dia da corrida
            cond_str = ("ceu_limpo" if condicao=="seco"
                        else ("chuva_leve" if condicao=="umido" else "chuva_forte"))
            for h in range(11, 18):
                ts = datetime(d_fim.year, d_fim.month, d_fim.day, h, 0)
                precip = rnd(0,5) if condicao != "seco" else 0.0
                c.execute(
                    "INSERT INTO meteorologia "
                    "(id_evento,timestamp,temperatura_ar,temperatura_pista,"
                    "umidade_pct,pressao_hpa,velocidade_vento_kmh,"
                    "direcao_vento_graus,precipitacao_mm,condicao) "
                    "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                    (id_ev, ts.strftime("%Y-%m-%d %H:%M:%S"),
                     temp_ar+rnd(-2,2), temp_pi+rnd(-3,3),
                     umid+rnd(-5,5), pres+rnd(-1,1),
                     vento+rnd(-5,5), rnd_int(0,359), precip, cond_str))

            eventos_info.append((id_ev, id_corrida, apelido, condicao, lat, lon))
            data_ref += timedelta(days=14)

    return temp_ids, eventos_info


# =============================================================================
#  STEP 7 - CORRIDAS COMPLETAS
#  (setups, voltas, setores, pit_stops, resultados, gps_posicoes,
#   telemetria, telemetria_volta_resumo, alertas_pista,
#   incidentes, falhas, relatorios_engenharia)
# =============================================================================
def step_corridas(c, eventos_info, pilot_ids, eq_ids, veic_por_eq,
                  sensor_ids_veiculo, total_voltas, fazer_telemetria):

    print(f"  -> corridas: {len(eventos_info)} eventos x {total_voltas} voltas x {len(pilot_ids)} pilotos")

    c.execute("SELECT id_piloto,id_equipe FROM pilotos")
    piloto_eq = {r[0]:r[1] for r in c.fetchall()}

    pilotos_lista = list(pilot_ids.items())  # [(cod, id), ...]

    pontos_acc = {pid:0 for pid in pilot_ids.values()}
    stats      = {pid:{"vitorias":0,"poles":0,"podios":0,"vr":0}
                  for pid in pilot_ids.values()}

    piloto_list_ids = list(pilot_ids.values())  # para calcular idx

    def get_veic(pid):
        eq = piloto_eq.get(pid)
        vs = veic_por_eq.get(eq, [])
        if not vs:
            return None
        idx = piloto_list_ids.index(pid) % len(vs) if pid in piloto_list_ids else 0
        return vs[idx]

    for ev_idx,(id_ev,id_sessao,apelido,condicao,circ_lat,circ_lon) in enumerate(eventos_info):
        print(f"     [{ev_idx+1:02d}/{len(eventos_info)}] {apelido} ({condicao})")

        base_ms = TEMPO_BASE.get(apelido, 85000)

        grid = list(pilotos_lista)
        random.shuffle(grid)
        pole_id = grid[0][1]
        stats[pole_id]["poles"] += 1

        sc_volta = rnd_int(10, total_voltas-15) if random.random()<0.4 else None
        sc_dur   = rnd_int(3,6) if sc_volta else 0
        abandonam = {pid for _,pid in pilotos_lista if random.random()<0.10}

        # ── SETUPS ──────────────────────────────────────────────────────────
        for cod,pid in pilotos_lista:
            id_v = get_veic(pid)
            if not id_v or not id_sessao:
                continue
            comp_d = random.choice(COMPOSTOS_S)
            comp_t = random.choice(COMPOSTOS_S)
            c.execute(
                "INSERT INTO setups_veiculo "
                "(id_veiculo,id_sessao,id_piloto,"
                "asa_dianteira_graus,asa_traseira_graus,"
                "altura_dianteira_mm,altura_traseira_mm,"
                "mola_dianteira_nm,mola_traseira_nm,"
                "composto_dianteiro,composto_traseiro,"
                "pressao_pneu_dd_psi,pressao_pneu_de_psi,"
                "pressao_pneu_td_psi,pressao_pneu_te_psi,"
                "balanco_freio_pct,carga_combustivel_kg,observacoes) "
                "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                (id_v,id_sessao,pid,
                 rnd(8,15),rnd(12,22),rnd(18,26),rnd(55,68),
                 rnd(50000,90000),rnd(55000,95000),
                 comp_d,comp_t,
                 rnd(21,25),rnd(21,25),rnd(19,23),rnd(19,23),
                 rnd(54,62),rnd(100,110),
                 fake.sentence(nb_words=6)))

        # ── VOLTAS ──────────────────────────────────────────────────────────
        resultados_temp = []
        id_voltas_piloto = {}

        for pos_larg,(cod,pid) in enumerate(grid, 1):
            id_v = get_veic(pid)
            if not id_v:
                continue

            hab      = HABILIDADE.get(cod, 800)
            pits     = estrategia_pit(total_voltas, condicao)
            pits_map = {volta:(ci,co) for volta,ci,co in pits}

            if condicao == "seco":
                comp = random.choice(["slick_soft","slick_medium"])
            elif condicao == "umido":
                comp = "intermediario"
            else:
                comp = "chuva"

            combustivel = rnd(105,110)
            idade_pneu  = 0
            pos_atual   = pos_larg
            voltas_info = []  # (id_volta_db, t_ms, pit_in, comp_in, comp_out)
            abnd_volta  = rnd_int(total_voltas//4, total_voltas-5) if pid in abandonam else None

            for vn in range(1, total_voltas+1):
                if abnd_volta and vn > abnd_volta:
                    break

                sc_ativo = bool(sc_volta and sc_volta <= vn <= sc_volta + sc_dur)
                pit_in   = 1 if vn in pits_map else 0
                pit_out  = 1 if (vn-1) in pits_map else 0

                if pit_out and (vn-1) in pits_map:
                    _, comp = pits_map[vn-1]
                    idade_pneu = 0

                t_ms = gerar_tempo_volta(base_ms,comp,idade_pneu,combustivel,hab,sc_ativo)
                if pit_in:
                    t_ms += rnd_int(18000,28000)

                combustivel = max(combustivel - rnd(1.5,2.2), 0)
                idade_pneu += 1
                valida = 0 if (sc_ativo or pit_in or pit_out) else 1

                c.execute(
                    "INSERT INTO voltas "
                    "(id_sessao,id_piloto,id_veiculo,numero_volta,tempo_ms,valida,"
                    "pit_in,pit_out,composto_pneu,idade_pneu_voltas,"
                    "combustivel_kg,posicao_inicio,posicao_fim) "
                    "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                    (id_sessao,pid,id_v,vn,t_ms,valida,
                     pit_in,pit_out,comp,idade_pneu,
                     round(combustivel,2),pos_atual,pos_atual))
                id_vv = c.lastrowid

                # setores (3 por volta)
                for sn,frac in enumerate([0.30,0.38,0.32], 1):
                    c.execute(
                        "INSERT INTO setores_volta "
                        "(id_volta,numero_setor,tempo_ms,velocidade_max_kmh) "
                        "VALUES (%s,%s,%s,%s)",
                        (id_vv, sn,
                         int(t_ms*frac + rnd_int(-500,500)),
                         rnd(200,360)))

                ci_out = pits_map[vn][0] if pit_in else None
                co_out = pits_map[vn][1] if pit_in else None
                voltas_info.append((id_vv, t_ms, pit_in, ci_out, co_out))

            id_voltas_piloto[pid] = voltas_info

            # pit stops
            for id_vv,_,pit_in,ci,co in voltas_info:
                if pit_in and ci:
                    c.execute(
                        "INSERT INTO pit_stops "
                        "(id_volta,duracao_ms,troca_pneu,composto_entrada,composto_saida) "
                        "VALUES (%s,%s,1,%s,%s)",
                        (id_vv, rnd_int(19000,27000), ci, co))

            t_total = sum(t for _,t,*_ in voltas_info)
            resultados_temp.append(
                (t_total, pos_larg, pid, len(voltas_info), pid in abandonam))

        # ── RESULTADOS ──────────────────────────────────────────────────────
        resultados_temp.sort(key=lambda x: (x[4], -x[3], x[0]))
        lider_t = resultados_temp[0][0] if resultados_temp else 1

        for pos_fin,(t_tot,pos_larg,pid,completou,abnd) in enumerate(resultados_temp, 1):
            id_v = get_veic(pid)
            if not id_v:
                continue
            status = "abandonou" if abnd else "finalizado"
            motivo = random.choice([
                "Problema mecanico","Acidente","Falha eletrica",
                "Pressao oleo","Sobreaquecimento"
            ]) if abnd else None
            pts = PONTOS_F1[pos_fin-1] if pos_fin<=10 and not abnd else 0

            c.execute(
                "INSERT INTO resultados "
                "(id_sessao,id_piloto,id_veiculo,posicao_largada,posicao_final,"
                "voltas_completadas,tempo_total_ms,diferenca_lider_ms,"
                "pontos,volta_rapida,status_corrida,motivo_abandono) "
                "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,0,%s,%s)",
                (id_sessao,pid,id_v,pos_larg,pos_fin,
                 completou,t_tot,t_tot-lider_t,pts,status,motivo))

            pontos_acc[pid] = pontos_acc.get(pid,0) + pts
            if pos_fin==1 and not abnd:
                stats[pid]["vitorias"] += 1
            if pos_fin<=3 and not abnd:
                stats[pid]["podios"]   += 1

        # ── GPS POSICOES (top 3 finais, ate 8 voltas cada) ──────────────────
        top3 = [pid for _,_,pid,_,abnd in resultados_temp[:3] if not abnd]
        for pid in top3:
            id_v = get_veic(pid)
            if not id_v:
                continue
            voltas_info = id_voltas_piloto.get(pid, [])[:8]
            ts_ms = 0
            for id_vv,t_ms,*_ in voltas_info:
                n_pts = 30  # ~30 pontos GPS por volta
                for i in range(n_pts):
                    angulo = 2 * 3.14159 * i / n_pts
                    lat_gps = circ_lat + 0.005 * round(angulo, 4)
                    lon_gps = circ_lon + 0.008 * round(angulo, 4)
                    c.execute(
                        "INSERT INTO gps_posicoes "
                        "(id_sessao,id_veiculo,timestamp_ms,latitude,longitude,"
                        "altitude_m,velocidade_kmh,direcao_graus,precisao_m) "
                        "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                        (id_sessao, id_v,
                         ts_ms + i*(t_ms//n_pts),
                         round(lat_gps,7), round(lon_gps,7),
                         rnd(0,10), rnd(60,360), rnd(0,360), rnd(0.5,3)))
                ts_ms += t_ms

        # ── TELEMETRIA BRUTA + RESUMO (vencedor, 5 voltas, 6 sensores) ──────
        if fazer_telemetria and resultados_temp:
            venc_pid  = resultados_temp[0][2]
            id_v_venc = get_veic(venc_pid)
            if id_v_venc and id_v_venc in sensor_ids_veiculo:
                sens_ids    = sensor_ids_veiculo[id_v_venc][:6]
                voltas_venc = id_voltas_piloto.get(venc_pid,[])[:5]

                tel_rows = []
                ts_ms = 0
                for id_vv,t_ms,*_ in voltas_venc:
                    for tick in range(0, 90000, 2000):
                        for sid in sens_ids:
                            tel_rows.append((
                                id_sessao, id_vv, id_v_venc, venc_pid, sid,
                                ts_ms+tick, round(random.uniform(100,15000),4), 100))
                    ts_ms += 90000

                if tel_rows:
                    c.executemany(
                        "INSERT INTO telemetria "
                        "(id_sessao,id_volta,id_veiculo,id_piloto,id_sensor,"
                        "timestamp_ms,valor,qualidade) "
                        "VALUES (%s,%s,%s,%s,%s,%s,%s,%s)", tel_rows)

                # telemetria_volta_resumo
                for id_vv,t_ms,*_ in voltas_venc:
                    c.execute(
                        "INSERT IGNORE INTO telemetria_volta_resumo "
                        "(id_volta,"
                        "rpm_medio,rpm_max,"
                        "temperatura_motor_media,temperatura_motor_max,"
                        "velocidade_media_kmh,velocidade_max_kmh,"
                        "temperatura_freio_dd_media,temperatura_freio_de_media,"
                        "temperatura_freio_td_media,temperatura_freio_te_media,"
                        "pressao_freio_max_bar,"
                        "temperatura_pneu_dd_media,temperatura_pneu_de_media,"
                        "temperatura_pneu_td_media,temperatura_pneu_te_media,"
                        "g_longitudinal_max,g_lateral_max,"
                        "consumo_combustivel_kg,"
                        "energia_recuperada_kj,energia_implantada_kj,"
                        "angulo_volante_medio) "
                        "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                        (id_vv,
                         rnd(10000,14000), rnd(14000,15500),
                         rnd(85,120),      rnd(130,190),
                         rnd(180,280),     rnd(320,380),
                         rnd(400,800),     rnd(400,800),
                         rnd(300,600),     rnd(300,600),
                         rnd(50,150,3),
                         rnd(70,110),      rnd(70,110),
                         rnd(80,120),      rnd(80,120),
                         rnd(3,6,3),       rnd(3,5,3),
                         rnd(1.5,2.5,3),
                         rnd(200,500,3),   rnd(150,400,3),
                         rnd(-45,45,3)))

        # ── ALERTAS DE PISTA ────────────────────────────────────────────────
        if sc_volta:
            ts0 = sc_volta * base_ms
            ts1 = (sc_volta + sc_dur) * base_ms
            c.execute(
                "INSERT INTO alertas_pista "
                "(id_sessao,tipo,timestamp_inicio_ms,timestamp_fim_ms,motivo) "
                "VALUES (%s,'safety_car',%s,%s,'Incidente em pista')",
                (id_sessao, ts0, ts1))
            c.execute(
                "INSERT INTO alertas_pista "
                "(id_sessao,tipo,timestamp_inicio_ms,timestamp_fim_ms,motivo) "
                "VALUES (%s,'amarelo',%s,%s,'Detritos na pista')",
                (id_sessao, ts0-5000, ts0))

        c.execute(
            "INSERT INTO alertas_pista "
            "(id_sessao,tipo,timestamp_inicio_ms,motivo) "
            "VALUES (%s,'drs_habilitado',%s,'DRS liberado apos volta de formacao')",
            (id_sessao, 200000))

        # ── INCIDENTES ──────────────────────────────────────────────────────
        tipos_inc = ["colisao","toque","ultrapassagem_ilegal","ignorar_bandeira",
                     "excesso_velocidade_pit","parametro_tecnico","outro"]
        pens_inc  = ["nenhuma","advertencia","5_segundos","10_segundos",
                     "drive_through","stop_and_go"]
        for _ in range(rnd_int(1, 4)):
            _,pid_i = random.choice(pilotos_lista)
            c.execute(
                "INSERT INTO incidentes "
                "(id_sessao,id_piloto,tipo,descricao,penalidade,volta_ocorrencia) "
                "VALUES (%s,%s,%s,%s,%s,%s)",
                (id_sessao, pid_i,
                 random.choice(tipos_inc),
                 fake.sentence(nb_words=7),
                 random.choice(pens_inc),
                 rnd_int(1, total_voltas)))

        # ── FALHAS ──────────────────────────────────────────────────────────
        sistemas = ["motor","freios","suspensao","eletrico",
                    "cambio","ERS","refrigeracao"]
        descrs   = [
            "Temperatura acima do limite",
            "Pressao fora do range operacional",
            "Leitura intermitente do sensor",
            "Vibracao anormal detectada",
            "Queda de tensao na bateria ERS",
            "Vazamento detectado no oleo",
            "Falha de comunicacao CAN bus",
        ]
        for _ in range(rnd_int(0, 4)):
            _,pid_f = random.choice(pilotos_lista)
            eq  = piloto_eq.get(pid_f)
            vs  = veic_por_eq.get(eq, [])
            id_vf = vs[0] if vs else None
            if id_vf:
                c.execute(
                    "INSERT INTO falhas "
                    "(id_sessao,id_veiculo,timestamp_ms,tipo,sistema,descricao,"
                    "valor_lido,valor_esperado,resolvida) "
                    "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                    (id_sessao, id_vf,
                     rnd_int(500000,5000000),
                     random.choice(["critica","alta","media","baixa"]),
                     random.choice(sistemas),
                     random.choice(descrs),
                     rnd(0,200), rnd(0,150), rnd_int(0,1)))

        # ── RELATORIOS DE ENGENHARIA ─────────────────────────────────────────
        amostra = random.sample(pilotos_lista, min(3, len(pilotos_lista)))
        for _,(cod,pid) in enumerate(amostra):
            eng = random.choice(ENGENHEIROS)
            c.execute(
                "INSERT INTO relatorios_engenharia "
                "(id_sessao,id_piloto,engenheiro,tipo,titulo,conteudo,versao) "
                "VALUES (%s,%s,%s,%s,%s,%s,1)",
                (id_sessao, pid, eng, "corrida",
                 f"Analise de Corrida GP {apelido} - {cod}",
                 (f"RESUMO: Piloto {cod} completou {total_voltas} voltas. "
                  f"Estrategia de pneus executada conforme planejado. "
                  f"Temperatura media do motor dentro dos parametros. "
                  f"Consumo medio: {rnd(1.5,2.2)} kg/volta. "
                  f"Notas adicionais: {fake.sentence(nb_words=10)}")))

    return pontos_acc, stats


# =============================================================================
#  STEP 8 - USO DE COMPONENTES
# =============================================================================
def step_uso_componentes(c, temp_ids, pilot_ids):
    print("  -> uso_componentes")
    c.execute("SELECT id_componente FROM componentes")
    comp_ids = [r[0] for r in c.fetchall()]

    for t_idx,id_temp in enumerate(temp_ids):
        for cod,pid in pilot_ids.items():
            for id_comp in comp_ids:
                ns   = f"COMP-{pid:03d}-{id_comp:02d}-{rnd_int(100,999)}"
                km   = rnd(500,4500)
                hs   = rnd(1,10)
                pen  = 1 if km > 3800 else 0
                ano  = 2024 + t_idx
                c.execute(
                    "INSERT IGNORE INTO uso_componentes "
                    "(id_piloto,id_temporada,id_componente,numero_serie,"
                    "data_introducao,km_acumulado,horas_acumuladas,"
                    "penalidade_grid,status) "
                    "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                    (pid, id_temp, id_comp, ns,
                     f"{ano}-03-01",
                     round(km,2), round(hs,2), pen,
                     random.choice(["em_uso","reserva","descartado"])))


# =============================================================================
#  STEP 9 - CLASSIFICACAO DO CAMPEONATO
# =============================================================================
def step_classificacao(c, temp_ids, pilot_ids, piloto_eq, pontos_acc, stats):
    print("  -> classificacao_campeonato")
    ranking = sorted(pontos_acc.items(), key=lambda x: -x[1])
    for id_temp in temp_ids:
        for pos,(pid,pts) in enumerate(ranking, 1):
            eq = piloto_eq.get(pid)
            if not eq:
                continue
            st = stats.get(pid, {})
            c.execute(
                "INSERT IGNORE INTO classificacao_campeonato "
                "(id_temporada,id_piloto,id_equipe,posicao,pontos_total,"
                "vitorias,poles,podios,voltas_rapidas) "
                "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)",
                (id_temp, pid, eq, pos, round(pts,2),
                 st.get("vitorias",0), st.get("poles",0),
                 st.get("podios",0),   st.get("vr",0)))


# =============================================================================
#  STEP 10 - MANUTENCOES
# =============================================================================
def step_manutencoes(c, veic_por_eq):
    print("  -> manutencoes")
    tipos = ["preventiva","corretiva","revisao_pos_evento",
             "troca_componente","upgrade","inspecao"]
    todos = [v for vs in veic_por_eq.values() for v in vs]

    for id_v in todos:
        data_b = datetime(2026,3,1)
        for i in range(4):
            d0 = data_b + timedelta(days=14*i + rnd_int(0,3))
            d1 = d0 + timedelta(hours=rnd_int(4,48))
            ce = rnd(5000,150000)
            cr = ce * rnd(0.85,1.2)
            c.execute(
                "INSERT INTO manutencoes "
                "(id_veiculo,tipo,descricao,mecanico_responsavel,"
                "data_inicio,data_fim,custo_estimado,custo_real,status) "
                "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,'concluida')",
                (id_v, random.choice(tipos),
                 fake.sentence(nb_words=8),
                 random.choice(MECANICOS),
                 d0.strftime("%Y-%m-%d %H:%M:%S"),
                 d1.strftime("%Y-%m-%d %H:%M:%S"),
                 round(ce,2), round(cr,2)))


# =============================================================================
#  MAIN
# =============================================================================
def main():
    print("=" * 64)
    print("  APEX MOTORSPORT - Populador Completo (32 tabelas)")
    print(f"  Volume: {VOLUME.upper()}")
    print("=" * 64)

    conn = connect()
    c    = new_cur(conn)

    try:
        print("\n[1/10] Referencias basicas...")
        step_referencias(c)
        conn.commit()

        print("\n[2/10] Fabricantes, motores, equipes, pilotos...")
        eq_ids, pilot_ids, fab_ids = step_equipes_pilotos(c)
        conn.commit()

        print("\n[3/10] Circuitos...")
        circ_ids = step_circuitos(c)
        conn.commit()

        print("\n[4/10] Veiculos e modelos...")
        veic_por_eq = step_veiculos(c, eq_ids, fab_ids)
        conn.commit()

        print("\n[5/10] Sensores...")
        sensor_ids_veiculo = step_sensores(c, veic_por_eq)
        conn.commit()

        print("\n[6/10] Temporadas, eventos, sessoes, meteorologia...")
        temp_ids, eventos_info = step_temporadas_eventos(
            c, circ_ids, CFG["temporadas"], CFG["eventos_por_temp"])
        conn.commit()

        print("\n[7/10] Corridas completas...")
        c.execute("SELECT id_piloto,id_equipe FROM pilotos")
        piloto_eq = {r[0]:r[1] for r in c.fetchall()}

        pontos, stats = step_corridas(
            c, eventos_info, pilot_ids, eq_ids, veic_por_eq,
            sensor_ids_veiculo, CFG["voltas_corrida"], CFG["telemetria"])
        conn.commit()

        print("\n[8/10] Uso de componentes...")
        step_uso_componentes(c, temp_ids, pilot_ids)
        conn.commit()

        print("\n[9/10] Classificacao do campeonato...")
        step_classificacao(c, temp_ids, pilot_ids, piloto_eq, pontos, stats)
        conn.commit()

        print("\n[10/10] Manutencoes...")
        step_manutencoes(c, veic_por_eq)
        conn.commit()

        # ── RESUMO FINAL ─────────────────────────────────────────────────────
        print("\n" + "=" * 64)
        print("  CONCLUIDO! Registros por tabela:")
        print("=" * 64)

        todas = [
            "alertas_pista","canais_telemetria","categorias","circuitos",
            "classificacao_campeonato","componentes","equipes","eventos",
            "fabricantes","falhas","gps_posicoes","incidentes","manutencoes",
            "meteorologia","modelos_veiculo","motores","paises","pilotos",
            "pit_stops","relatorios_engenharia","resultados","sensores",
            "sessoes","setores_volta","setups_veiculo","telemetria",
            "telemetria_volta_resumo","temporadas","tipos_sensor",
            "uso_componentes","veiculos","voltas",
        ]
        total = 0
        for t in todas:
            c.execute(f"SELECT COUNT(*) FROM {t}")
            n = c.fetchone()[0]
            total += n
            status = "OK" if n > 0 else "!!"
            print(f"  [{status}] {t:<35} {n:>8} registros")

        print(f"\n  TOTAL GERAL: {total:,} registros em 32 tabelas")
        print("=" * 64)
        print("\n  Banco pronto para o Power BI! 🏁\n")

    except Exception as e:
        conn.rollback()
        print(f"\n ERRO: {e}")
        import traceback; traceback.print_exc()
    finally:
        c.close()
        conn.close()


if __name__ == "__main__":
    main()

  APEX MOTORSPORT - Populador Completo (32 tabelas)
  Volume: MEDIO

[1/10] Referencias basicas...
  -> paises, categorias, tipos_sensor, componentes, canais_telemetria

[2/10] Fabricantes, motores, equipes, pilotos...
  -> fabricantes, motores, equipes, pilotos

[3/10] Circuitos...
  -> circuitos

[4/10] Veiculos e modelos...
  -> modelos_veiculo, veiculos

[5/10] Sensores...
  -> sensores

[6/10] Temporadas, eventos, sessoes, meteorologia...
  -> temporadas, eventos, sessoes, meteorologia

[7/10] Corridas completas...
  -> corridas: 30 eventos x 60 voltas x 20 pilotos
     [01/30] Montreal (seco)
     [02/30] Budapest (seco)
     [03/30] Suzuka (molhado)
     [04/30] Monza (molhado)
     [05/30] Xangai (seco)
     [06/30] Interlagos (molhado)
     [07/30] Singapura (molhado)
     [08/30] Spa (seco)
     [09/30] Barcelona (umido)
     [10/30] Mexico (seco)
     [11/30] Monaco (seco)
     [12/30] Bahrein (seco)
     [13/30] Austin (seco)
     [14/30] Abu Dhabi (seco)
     [15/30] Silve